In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
#: Every module the pipeline imports. A module missing from this list is not
#: reported until some later cell dies with a bare ImportError, which is how
#: cluster_tier / weak_labels / dataset_builder went missing twice.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "cluster_tier": None,
    "fire_labels": None,
    "weak_labels": None,
    "dataset_builder": None,
    "training_design": None,
    "feature_gap": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import cluster_tier as ct
import dataset_builder as db
import feature_gap as fgap
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf
import training_design as td
import weak_labels as weak


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial block does not match the ledger")
if ct.N_CLUSTER_TIER + len(weak.LABEL_INPUT_FEATURES) < 1:
    problems.append("cluster_tier / weak_labels look empty")
if len(fl.FEATURES_141) != 141:
    problems.append(f"ledger has {len(fl.FEATURES_141)} features, expected 141")
if not hasattr(db, "SITE_DEG") or not hasattr(fgap, "gap_report"):
    problems.append("dataset_builder / feature_gap are stale")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print(f"  weak labels : {len(weak.LABEL_INPUT_FEATURES)} rule inputs published")
print(f"  site grid   : {db.SITE_DEG} deg")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

BOOTSTRAP


Saving lineage_features.py to lineage_features.py
Saving industrial_features.py to industrial_features.py
Saving feature_ledger.py to feature_ledger.py
Saving weak_labels.py to weak_labels.py
Saving training_design.py to training_design.py
Saving fire_labels.py to fire_labels.py
Saving feature_gap.py to feature_gap.py
Saving dataset_builder.py to dataset_builder.py
Saving colab_utils.py to colab_utils.py
Saving cluster_tier.py to cluster_tier.py
upload dialog returned 10 file(s): ['cluster_tier.py', 'colab_utils.py', 'dataset_builder.py', 'feature_gap.py', 'feature_ledger.py', 'fire_labels.py', 'industrial_features.py', 'lineage_features.py', 'training_design.py', 'weak_labels.py']
  feature_ledger         <- upload (31,302 bytes)
  industrial_features    <- upload (19,120 bytes)
  lineage_features       <- upload (34,759 bytes)
  cluster_tier           <- upload (21,397 bytes)
  fire_labels            <- upload (19,486 bytes)
  weak_labels            <- upload (18,567 bytes)
  dataset

In [4]:
# ============================================================================
# P4a TIER 1 -- 34 lineage features for every event, from clusters/ + membership
#
# This is the cheapest real feature pass in the project: it reads 532 MB of
# cluster-level aggregates and never touches the 15.4M-row detection table.
#
# Inputs
#   clusters/clusters_YYYY-MM-DD.parquet          5,337 files, 532 MB
#       date local_cluster_id start_time end_time n_detections centroid_x/y
#       total_frp mean_frp max_frp n_observation_times duration_hours
#       satellites n_satellites max_radius_m max_radius_km centroid_lon/lat
#   events/event_local_cluster_membership.parquet  7,589,305 rows
#       event_id  local_cluster_id  date
#
# Output
#   features/lineage/lineage_tier1_chunk_NNNN.parquet
#       event_id + the 34 CLUSTER_TIER_FEATURES, ledger order
#
# INSPECT_ONLY = True by default: prints the real schemas, verifies the join on
# one small window, writes nothing. A join that silently matches nothing looks
# exactly like a successful run, so that has to be checked before the full pass.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import feature_ledger as fl
import lineage_features as lf

INSPECT_ONLY = False
DATE_BATCH = 60           # days of clusters per read
EVENT_CHUNK = 400_000     # events per written chunk
VERIFY_DAYS = 3           # days used for the pre-flight join check

MODE = "INSPECT ONLY (writes nothing)" if INSPECT_ONLY else "FULL PASS (writes)"
print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   lineage {lf.N_LINEAGE}   "
      f"tier1 {ct.N_CLUSTER_TIER}")
print(f"MODE: {MODE}")
print("=" * 74)
print(cu.drive_report())

# ---------------------------------------------------------------------------
# 1. RESOLVE INPUTS
# ---------------------------------------------------------------------------

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No directory under {cu.MYDRIVE} contains {cu.CANONICAL_SUBPATH}."
    )
_CANON = PROJECT_ROOT / "full_2012_2026"

CLUSTERS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "clusters")
EVENTS_DIR, _ = cu.pick_by_file_count(PROJECT_ROOT, "events")
OUT_DIR = _CANON / "features" / "lineage"
QC_DIR = PROJECT_ROOT / "qc"

if CLUSTERS_DIR is None or EVENTS_DIR is None:
    raise FileNotFoundError("clusters/ or events/ not found under the project root")

MEMBERSHIP = next(
    (
        EVENTS_DIR / n
        for n in (
            "event_local_cluster_membership.parquet",
            "local_cluster_membership.parquet",
        )
        if cu.path_ok(EVENTS_DIR / n)
    ),
    None,
)
if MEMBERSHIP is None:
    print("parquet files under", EVENTS_DIR)
    for p in sorted(EVENTS_DIR.glob("*.parquet"))[:20]:
        print("   ", p.name)
    raise FileNotFoundError(
        "Set MEMBERSHIP by hand to the event<->cluster mapping, then re-run."
    )

print()
print("RESOLVED LOCATIONS")
print("-" * 74)
for label, path in (
    ("clusters", CLUSTERS_DIR),
    ("events", EVENTS_DIR),
    ("membership", MEMBERSHIP),
    ("output", OUT_DIR),
    ("qc", QC_DIR),
):
    flag = "ok " if cu.path_ok(path) else "new"
    print(f"  [{flag}] {label:12s} {path}")
print("-" * 74)

# ---------------------------------------------------------------------------
# 2. INDEX THE DAILY FILES
# ---------------------------------------------------------------------------

CLUSTER_INDEX, cluster_stats = lf.index_daily_files(CLUSTERS_DIR, prefix="clusters")
if not CLUSTER_INDEX:
    raise FileNotFoundError(f"No dated cluster files under {CLUSTERS_DIR}")
ordered_dates = sorted(CLUSTER_INDEX)
print()
print(f"daily cluster files : {len(CLUSTER_INDEX):,}  "
      f"{ordered_dates[0]} .. {ordered_dates[-1]}")
print(f"  {cluster_stats}")

# ---------------------------------------------------------------------------
# 3. MEMBERSHIP
# ---------------------------------------------------------------------------

t0 = time.time()
membership = pq.read_table(MEMBERSHIP).to_pandas()
print()
print(f"membership rows : {len(membership):,}  "
      f"cols={list(membership.columns)}  ({time.time() - t0:.1f}s)")

MEM_EVENT_COL = next(
    (c for c in ("event_id", "global_event_id") if c in membership.columns), None
)
MEM_CLUSTER_COL = next(
    (c for c in ("local_cluster_id", "cluster_id") if c in membership.columns), None
)
if MEM_EVENT_COL is None or MEM_CLUSTER_COL is None:
    raise KeyError(
        f"membership needs an event id and a cluster id; has "
        f"{list(membership.columns)}"
    )
MEM_DATE_COL = next(
    (c for c in ("date", "acq_date", "cluster_date") if c in membership.columns), None
)
print(f"  event col   : {MEM_EVENT_COL}")
print(f"  cluster col : {MEM_CLUSTER_COL}")
print(f"  date col    : {MEM_DATE_COL or '(none -- will derive from cluster id)'}")

n_events_total = membership[MEM_EVENT_COL].nunique()
print(f"  distinct events : {n_events_total:,}")

membership[MEM_CLUSTER_COL] = membership[MEM_CLUSTER_COL].astype(str)
member_index = membership.set_index(MEM_CLUSTER_COL)

# ---------------------------------------------------------------------------
# 4. SCHEMA + PRE-FLIGHT JOIN CHECK
# ---------------------------------------------------------------------------

sample_day = ordered_dates[0]
sample = pd.read_parquet(CLUSTER_INDEX[sample_day])
print()
print("=" * 74)
print(f"SAMPLE CLUSTER FILE  {CLUSTER_INDEX[sample_day].name}  "
      f"rows={len(sample):,}")
print("=" * 74)
for col, typ in pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow.types and [
    (f.name, str(f.type)) for f in
    pq.ParquetFile(CLUSTER_INDEX[sample_day]).schema_arrow
]:
    print(f"  {col:26s} {typ}")

sample_cluster_col = next(
    (
        c
        for c in (
            MEM_CLUSTER_COL, "local_cluster_id", "cluster_id", "local_cluster"
        )
        if c in sample.columns
    ),
    None,
)
if sample_cluster_col is None:
    raise KeyError(
        f"No cluster id column in {sample_day}. Columns: {list(sample.columns)}"
    )
print()
print(f"  cluster id column : {sample_cluster_col}")

# THE check that matters: does the membership table actually cover these
# clusters? If it does not, every join below produces nothing while looking
# like a clean run.
sample_ids = sample[sample_cluster_col].astype(str)
hits = sample_ids.isin(member_index.index)
print()
print("=" * 74)
print("PRE-FLIGHT JOIN CHECK")
print("=" * 74)
print(f"  clusters in this file        : {len(sample_ids):,}")
print(f"  found in membership         : {int(hits.sum()):,}")
print(f"  unmatched                   : {int((~hits).sum()):,}")
if not hits.any():
    raise RuntimeError(
        f"ZERO of {len(sample_ids):,} cluster ids in {sample_day} appear in "
        f"{MEMBERSHIP.name}. The two tables are not joinable as-is -- check "
        "whether the id formats differ (e.g. '20120120_0' vs '2012-01-20_0')."
    )
if (~hits).any():
    print(f"  unmatched examples          : "
          f"{list(sample_ids[~hits])[:5]}")
    print(f"  membership examples         : "
          f"{list(member_index.index[:3])}")
    print("  -> a partial match is expected if this day is the first one;")
    print("     if the unmatched share is large, STOP and compare id formats.")

# ---------------------------------------------------------------------------
# 5. INSPECT-ONLY EXIT
# ---------------------------------------------------------------------------

if INSPECT_ONLY:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    report = {
        "clusters_dir": str(CLUSTERS_DIR),
        "n_daily_files": len(CLUSTER_INDEX),
        "first": ordered_dates[0],
        "last": ordered_dates[-1],
        "cluster_index_stats": cluster_stats,
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "membership_event_col": MEM_EVENT_COL,
        "membership_cluster_col": MEM_CLUSTER_COL,
        "membership_date_col": MEM_DATE_COL,
        "distinct_events": int(n_events_total),
        "sample_file": str(CLUSTER_INDEX[sample_day].name),
        "sample_cluster_col": sample_cluster_col,
        "preflight_matched": int(hits.sum()),
        "preflight_unmatched": int((~hits).sum()),
        "output_features": ct.CLUSTER_TIER_FEATURES,
    }
    (QC_DIR / "p4a_tier1_schema.json").write_text(
        json.dumps(report, indent=2)
    )
    print()
    print(f"matched {int(hits.sum()):,} / {len(sample_ids):,}  "
          "-- join is viable")
    print(f"Wrote {QC_DIR / 'p4a_tier1_schema.json'}")
    print()
    print("INSPECTION COMPLETE -- nothing written to features/.")
    print("------------------------------------------------------------")
    print("STOPPED IN INSPECT MODE. Nothing was written.")
    print("To actually run: edit INSPECT_ONLY to False, then re-run.")
    print("------------------------------------------------------------")
    _STOP = True
else:
    _STOP = False

# ===========================================================================
# 6. FULL PASS
# ===========================================================================

def run_full_pass():
    """Execute the whole pass. Called only when INSPECT_ONLY is False."""
    print()
    print("=" * 74)
    print("P4a TIER 1 -- FULL PASS  (writing to features/lineage/)")
    print("=" * 74)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    batch_size = max(1, int(DATE_BATCH))

    null_counts = {name: 0 for name in ct.CLUSTER_TIER_FEATURES}
    rows_out = 0
    chunk_files = []
    events_seen = set()
    unmatched_clusters = 0
    clusters_read = 0

    for batch_no, start in enumerate(range(0, len(ordered_dates), batch_size)):
        window = ordered_dates[start:start + batch_size]

        tables = []
        for day in window:
            path = CLUSTER_INDEX[day]
            if not cu.path_ok(path):
                print(f"  !! unreadable {path.name}")
                continue
            tables.append(pq.read_table(path))
        if not tables:
            continue
        det = pa.concat_tables(tables).to_pandas()
        del tables

        clusters_read += len(det)
        det[sample_cluster_col] = det[sample_cluster_col].astype(str)

        # Keep only clusters the membership table knows about, and say how many
        # were dropped: an unexpected number here means the join keys disagree.
        known = det[sample_cluster_col].isin(member_index.index)
        n_unknown = int((~known).sum())
        if n_unknown:
            unmatched_clusters += n_unknown
            if batch_no == 0:
                print(f"  batch 0: {n_unknown:,} of {len(det):,} clusters are not "
                      f"in membership (expected only if this is the first day)")
        det = det[known]
        if det.empty:
            continue

        # The membership table carries its own `date`, which collides with the
        # cluster file's `date` and makes the join raise outright. Bring across
        # only what is actually needed.
        bring = [MEM_EVENT_COL]
        if MEM_DATE_COL and MEM_DATE_COL not in det.columns:
            bring.append(MEM_DATE_COL)
        elif MEM_DATE_COL:
            # Same name on both sides: keep the cluster file's own value and skip
            # the duplicate rather than failing on an overlapping column.
            print("  note: membership date column shares a name with the cluster "
                  "file; using the cluster file's value")

        joined = det.join(
            member_index[bring], on=sample_cluster_col, how="inner", rsuffix="_mem"
        )
        if joined.empty:
            print(f"  !! batch {batch_no}: join produced 0 rows -- STOP")
            break

        computed = ct.compute_cluster_tier_features(joined, event_col=MEM_EVENT_COL)
        # One row per event: an event spans many daily batches, so later batches
        # would otherwise re-emit events already written.
        fresh = computed[~computed.index.isin(events_seen)]
        if fresh.empty:
            continue
        events_seen.update(fresh.index.tolist())

        for name in ct.CLUSTER_TIER_FEATURES:
            null_counts[name] += int(fresh[name].isna().sum())

        out = fresh.reset_index()
        path_out = OUT_DIR / f"lineage_tier1_chunk_{batch_no:04d}.parquet"
        out.to_parquet(path_out, index=False, compression="snappy")
        chunk_files.append({"file": path_out.name, "rows": len(out)})
        rows_out += len(out)

        if batch_no % 5 == 0:
            print(f"  batch {batch_no:>4d}  days={len(window):>3d}  "
                  f"events={rows_out:>10,}  {time.time() - t0:>7.1f}s")

    # ---------------------------------------------------------------------------
    # 7. QC
    # ---------------------------------------------------------------------------

    print()
    print("=" * 74)
    print("P4a TIER 1 -- QC")
    print("=" * 74)
    print(f"cluster rows read      : {clusters_read:,}")
    print(f"clusters not in member : {unmatched_clusters:,}")
    print(f"events written         : {rows_out:,}  of {n_events_total:,} in "
          f"membership ({100.0 * rows_out / max(n_events_total, 1):.1f}%)")

    assert list(features_cols := [c for c in ct.CLUSTER_TIER_FEATURES]) == \
        ct.CLUSTER_TIER_FEATURES
    print(f"feature columns        : {len(features_cols)}")

    missing = n_events_total - rows_out
    if missing > 0:
        print()
        print(f"  NOTE: {missing:,} membership events produced no features. That is")
        print("  expected only if those events have no clusters on disk. Events with")
        print("  a single cluster still produce FRP/count/duration values; only the")
        print("  trend and shape features are NaN for them.")

    print()
    print(f"{'feature':32s} {'NaN %':>8s}")
    summary = {}
    for name in ct.CLUSTER_TIER_FEATURES:
        frac = null_counts[name] / rows_out if rows_out else float("nan")
        summary[name] = frac
        flag = "  <-- check" if frac > 0.6 and name not in ct.DEFERRED_REASONS else ""
        print(f"  {name:30s} {frac:>7.2%}{flag}")

    manifest = {
        "clusters_dir": str(CLUSTERS_DIR),
        "membership": str(MEMBERSHIP),
        "membership_rows": int(len(membership)),
        "events_in_membership": int(n_events_total),
        "events_written": int(rows_out),
        "cluster_rows_read": int(clusters_read),
        "clusters_not_in_membership": int(unmatched_clusters),
        "date_batch": batch_size,
        "tier": 1,
        "feature_order": ct.CLUSTER_TIER_FEATURES,
        "nan_fraction": summary,
        "deferred_features": list(ct.DEFERRED_REASONS),
        "chunks": chunk_files,
    }
    (OUT_DIR / "manifest_tier1.json").write_text(json.dumps(manifest, indent=2))
    print()
    print("Wrote", OUT_DIR / "manifest_tier1.json")

    print()
    print("NEXT")
    print("  1. Read the NaN table. A high NaN share on a trend feature usually")
    print("     means single-cluster events, which is correct. A high NaN share on")
    print("     FRP_MEDIAN or DETECTION_COUNT_LOG means a broken join.")
    print("  2. Run colab_p3a_industrial.py for the 18 industrial features.")
    print("  3. Join tier1 + industrial -> 52 features, then label and train.")
    print("  4. The 20 deferred features are NOT here by accident -- see")
    print("     cluster_tier.DEFERRED_REASONS for the reason each one is absent.")


if not _STOP:
    run_full_pass()


ledger 1.3.0   lineage 54   tier1 34
MODE: FULL PASS (writes)
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED LOCATIONS
--------------------------------------------------------------------------
  [

KeyboardInterrupt: 

In [6]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import training_design as td

INSPECT_ONLY = True
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
WEAK_VARIANT = [weak.WIDE, weak.TIGHT]
WEAK_VARIANT_NAMES = ["wide(1-10 km)", "tight(2/5/10 m)"]

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        weak_out = weak.assign_weak_labels(joined, WEAK_VARIANT[0])
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_weak"] = weak_out["label"]
        joined["label_weak_rule"] = weak_out["rule"]
        joined["label_weak_source"] = weak_out["label_source"]
        for cls, n_ in Counter(weak_out["label"]).items():
            weak_counts[cls] += n_
        # Count DISTINCT events blocked by each input. Summing the per-string
        # Counter double-counts any input noted by two rules, which reported
        # 14,531,698 "events blocked" for a 7,265,849-event table.
        for event_id, entry in zip(joined["event_id"],
                                   label_out["missing_inputs"]):
            if entry:
                for nm in entry.split(","):
                    missing_pairs.setdefault(nm, set()).add(event_id)
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "label_weak", "label_weak_rule", "label_weak_source"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS (industrial-context hierarchy; manufactured from")
    print("  features the model sees, so never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm in sorted(missing_pairs, key=lambda k: -len(missing_pairs[k])):
            n = len(missing_pairs[nm])
            print(f"    {nm:32s} {n:>10,} of {total:,} events blocked "
                  f"({100.0 * n / total:.1f}%)")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: len(v) for k, v in missing_pairs.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_variant": WEAK_VARIANT_NAMES[0],
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

In [7]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
WEAK_VARIANT = [weak.WIDE, weak.TIGHT]
WEAK_VARIANT_NAMES = ["wide(1-10 km)", "tight(2/5/10 m)"]

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        weak_out = weak.assign_weak_labels(joined, WEAK_VARIANT[0])
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_weak"] = weak_out["label"]
        joined["label_weak_rule"] = weak_out["rule"]
        joined["label_weak_source"] = weak_out["label_source"]
        for cls, n_ in Counter(weak_out["label"]).items():
            weak_counts[cls] += n_
        for entry, n_missing in Counter(
                label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "label_weak", "label_weak_rule", "label_weak_source"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS (industrial-context hierarchy; manufactured from")
    print("  features the model sees, so never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm, n in missing_counter.most_common(20):
            print(f"    {nm:32s} {n:>10,} events blocked")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: int(v) for k, v in missing_counter.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_variant": WEAK_VARIANT_NAMES[0],
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

In [8]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import training_design as td

INSPECT_ONLY = True
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
WEAK_VARIANT = [weak.WIDE, weak.TIGHT]
WEAK_VARIANT_NAMES = ["wide(1-10 km)", "tight(2/5/10 m)"]

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        weak_out = weak.assign_weak_labels(joined, WEAK_VARIANT[0])
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        joined["label_weak"] = weak_out["label"]
        joined["label_weak_rule"] = weak_out["rule"]
        joined["label_weak_source"] = weak_out["label_source"]
        for cls, n_ in Counter(weak_out["label"]).items():
            weak_counts[cls] += n_
        # Count DISTINCT events blocked by each input. Summing the per-string
        # Counter double-counts any input noted by two rules, which reported
        # 14,531,698 "events blocked" for a 7,265,849-event table.
        for event_id, entry in zip(joined["event_id"],
                                   label_out["missing_inputs"]):
            if entry:
                for nm in entry.split(","):
                    missing_pairs.setdefault(nm, set()).add(event_id)
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "label_weak", "label_weak_rule", "label_weak_source"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("y", y_frame, y_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # ---- phase 3: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS (industrial-context hierarchy; manufactured from")
    print("  features the model sees, so never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm in sorted(missing_pairs, key=lambda k: -len(missing_pairs[k])):
            n = len(missing_pairs[nm])
            print(f"    {nm:32s} {n:>10,} of {total:,} events blocked "
                  f"({100.0 * n / total:.1f}%)")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: len(v) for k, v in missing_pairs.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_variant": WEAK_VARIANT_NAMES[0],
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

In [3]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
#: Every module the pipeline imports. A module missing from this list is not
#: reported until some later cell dies with a bare ImportError, which is how
#: cluster_tier / weak_labels / dataset_builder went missing twice.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "cluster_tier": None,
    "fire_labels": None,
    "weak_labels": None,
    "dataset_builder": None,
    "training_design": None,
    "feature_gap": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import cluster_tier as ct
import dataset_builder as db
import feature_gap as fgap
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf
import training_design as td
import weak_labels as weak


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial block does not match the ledger")
if ct.N_CLUSTER_TIER + len(weak.LABEL_INPUT_FEATURES) < 1:
    problems.append("cluster_tier / weak_labels look empty")
if len(fl.FEATURES_141) != 141:
    problems.append(f"ledger has {len(fl.FEATURES_141)} features, expected 141")
if not hasattr(db, "SITE_DEG") or not hasattr(fgap, "gap_report"):
    problems.append("dataset_builder / feature_gap are stale")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print(f"  weak labels : {len(weak.LABEL_INPUT_FEATURES)} rule inputs published")
print(f"  site grid   : {db.SITE_DEG} deg")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

BOOTSTRAP


Saving rule_calibration.py to rule_calibration (1).py
Saving cluster_tier.py to cluster_tier (2).py
Saving colab_utils.py to colab_utils (3).py
Saving dataset_builder.py to dataset_builder (2).py
Saving feature_gap.py to feature_gap (2).py
Saving fire_labels.py to fire_labels (2).py
Saving lineage_features.py to lineage_features (2).py
Saving weak_labels.py to weak_labels (2).py
Saving training_design.py to training_design (2).py
Saving feature_ledger.py to feature_ledger (6).py
Saving industrial_features.py to industrial_features (3).py
upload dialog returned 11 file(s): ['cluster_tier (2).py', 'colab_utils (3).py', 'dataset_builder (2).py', 'feature_gap (2).py', 'feature_ledger (6).py', 'fire_labels (2).py', 'industrial_features (3).py', 'lineage_features (2).py', 'rule_calibration (1).py', 'training_design (2).py', 'weak_labels (2).py']
  feature_ledger         <- upload (31,302 bytes)
  industrial_features    <- upload (19,120 bytes)
  lineage_features       <- upload (34,759 bytes

In [4]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from dataclasses import asdict
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import rule_calibration as rc
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
#: Target class sizes used to calibrate the persistence thresholds. Chosen so a
#: class stays rare but learnable: ~0.05% of 7.27M is about 3,600 events.
CALIB_TARGETS = {
    weak.INDUSTRIAL_FLARE: 3_600,
    weak.INDUSTRIAL_FIRE: 400_000,
    weak.MINING: 60_000,
}

#: Rows sampled per partition for calibration. 16 x 12,500 = 200,000, ample
#: for percentiles in the 99.5th to 99.95th range.
CALIB_SAMPLE_PER_PARTITION = 12_500

#: Columns the weak rules read, plus the derived inputs they need.
CALIB_INPUT_COLUMNS = (
    "DIST_REFINERY_LOG", "DIST_NEAREST_FACILITY_LOG", "FACILITY_COUNT_5KM",
    "DIST_POWERPLANT_LOG", "DIST_PETROCHEMICAL_LOG", "DIST_STEEL_LOG",
    "DIST_CEMENT_LOG", "DIST_MINE_LOG", "FRP_PERSISTENCE",
    "DETECTION_COUNT_LOG", "TIME_TO_PEAK", "AREA_GROWTH_RATE",
    "SPREAD_SPEED_MEDIAN", "DIRECTION_CONSISTENCY", "INSIDE_FACILITY",
    "KNOWN_FLARE",
)

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    calib_parts = []
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    missing_counter = Counter()
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # ---- calibration sample -------------------------------------------
        # Thresholds must be calibrated on the WHOLE catalogue, but the pass
        # runs one partition at a time, so the calibration inputs are sampled
        # here and consumed in PHASE 3. Systematic sampling (every Nth row) so
        # the sample is not biased toward any partition's date range.
        _need = [c for c in CALIB_INPUT_COLUMNS if c in joined.columns]
        if _need:
            _step = max(1, len(joined) // CALIB_SAMPLE_PER_PARTITION)
            calib_parts.append(joined.iloc[::_step][_need + ["event_id"]])

        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        # Count DISTINCT events blocked by each input. Summing the per-string
        # Counter double-counts any input noted by two rules, which reported
        # 14,531,698 "events blocked" for a 7,265,849-event table.
        for event_id, entry in zip(joined["event_id"],
                                   label_out["missing_inputs"]):
            if entry:
                for nm in entry.split(","):
                    missing_pairs.setdefault(nm, set()).add(event_id)
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        for entry, n_missing in Counter(label_out["missing_inputs"]).items():
            if entry:
                for nm in entry.split(","):
                    missing_counter[nm] += n_missing
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        # `y` is written in PHASE 3, after calibration.
        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "missing_label_inputs"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # =================================================================
    # PHASE 3 -- calibrate the weak thresholds, then label
    #
    # Calibration needs the whole distribution, but the pass is partitioned, so
    # it happens between PHASE 2 and the label pass. Without this the
    # persistence thresholds stayed tied to the pre-fix scale: INDUSTRIAL_FLARE
    # fell 3,571 -> 0 and MINING 73,060 -> 61 in one re-run.
    # =================================================================
    print()
    print("=" * 74)
    print("PHASE 3 -- CALIBRATE THRESHOLDS, THEN LABEL")
    print("=" * 74)

    calib_sample = (
        pd.concat(calib_parts, ignore_index=True)
        if calib_parts
        else pd.DataFrame()
    )
    print(f"  calibration sample: {len(calib_sample):,} rows")

    print()
    print("  FRP_PERSISTENCE distribution (the input that moved scale):")
    print(rc.distribution(
        calib_sample, ["FRP_PERSISTENCE"]).to_string(index=False))

    thresholds, evidence = rc.calibrate_to_target_counts(
        calib_sample, CALIB_TARGETS, base=weak.WIDE)
    calib_table, calib_report, _ = rc.calibration_report(
        calib_sample, thresholds, evidence)
    print()
    print("  calibrated thresholds:")
    print(calib_table.to_string(index=False))
    print()
    print("  weak labels on the SAMPLE, before vs after calibration:")
    print(calib_report.to_string(index=False))

    (QC_DIR / "p6_rule_calibration.json").write_text(json.dumps(
        {"targets": CALIB_TARGETS,
         "thresholds": rc.thresholds_to_dict(thresholds),
         "evidence": [asdict(e) for e in evidence]},
        indent=2, default=str))
    print(f"  Wrote {QC_DIR}/p6_rule_calibration.json")

    # Label every partition with the calibrated thresholds.
    print()
    print("  labelling all partitions with the calibrated thresholds ...")
    y_writer = None
    labelled = 0
    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue
        j = (
            pd.read_parquet(base_p)
            .merge(pd.read_parquet(t1_p), on="event_id", how="inner")
            .merge(pd.read_parquet(in_p), on="event_id", how="inner")
        )
        if j.empty:
            continue
        lo = lab.assign_labels(j)
        wo = weak.assign_weak_labels(j, thresholds)
        for cls, n in Counter(wo["label"]).items():
            weak_counts[cls] += n
        yf = pd.DataFrame({
            "event_id": j["event_id"].to_numpy(),
            "label": lo["label"],
            "label_rule": lo["rule"],
            "label_confidence": lo["confidence"],
            "sample_weight": lab.label_weights(lo["label"], lo["confidence"]),
            "label_weak": wo["label"],
            "label_weak_rule": wo["rule"],
            "label_weak_source": wo["label_source"],
            "missing_label_inputs": lo["missing_inputs"],
        })
        if y_writer is None:
            y_writer = cu.LazyParquetWriter(OUT_DIR / "y.parquet")
        y_writer.write_frame(yf)
        labelled += len(yf)
        if part % 5 == 0:
            print(f"    partition {part:>2d}  labelled={labelled:>9,}")

    if y_writer is not None:
        y_writer.close()
    print(f"  y.parquet rows: {labelled:,}")
    if labelled != assembled["X"]:
        print(f"  !! y has {labelled:,} rows, expected "
              f"{assembled['X']:,} -- the label pass lost rows")

    # ---- phase 4: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS, CALIBRATED thresholds (industrial-context")
    print("  hierarchy; manufactured from features the model sees, so")
    print("  never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_counter:
        for nm in sorted(missing_pairs, key=lambda k: -len(missing_pairs[k])):
            n = len(missing_pairs[nm])
            print(f"    {nm:32s} {n:>10,} of {total:,} events blocked "
                  f"({100.0 * n / total:.1f}%)")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: len(v) for k, v in missing_pairs.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_thresholds": rc.thresholds_to_dict(thresholds),
        "weak_calibration_targets": CALIB_TARGETS,
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

KeyboardInterrupt: 

In [6]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from dataclasses import asdict
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import rule_calibration as rc
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
#: Target class sizes used to calibrate the persistence thresholds. Chosen so a
#: class stays rare but learnable: ~0.05% of 7.27M is about 3,600 events.
CALIB_TARGETS = {
    weak.INDUSTRIAL_FLARE: 3_600,
    weak.INDUSTRIAL_FIRE: 400_000,
    weak.MINING: 60_000,
}

#: Rows sampled per partition for calibration. 16 x 12,500 = 200,000, ample
#: for percentiles in the 99.5th to 99.95th range.
CALIB_SAMPLE_PER_PARTITION = 12_500

#: Columns the weak rules read, plus the derived inputs they need.
CALIB_INPUT_COLUMNS = (
    "DIST_REFINERY_LOG", "DIST_NEAREST_FACILITY_LOG", "FACILITY_COUNT_5KM",
    "DIST_POWERPLANT_LOG", "DIST_PETROCHEMICAL_LOG", "DIST_STEEL_LOG",
    "DIST_CEMENT_LOG", "DIST_MINE_LOG", "FRP_PERSISTENCE",
    "DETECTION_COUNT_LOG", "TIME_TO_PEAK", "AREA_GROWTH_RATE",
    "SPREAD_SPEED_MEDIAN", "DIRECTION_CONSISTENCY", "INSIDE_FACILITY",
    "KNOWN_FLARE",
)

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = ["SITE_FRP_BASELINE", "SITE_N_EVENTS",
                          "FRP_ANOMALY_VS_SITE", "MISSING_FEATURE_FRACTION"]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    calib_parts = []
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    # Distinct events blocked per missing input. A dict-of-sets, NOT a Counter:
    # summing per-string counts double-counted an input noted by two rules and
    # reported 14,531,698 blocked for a 7,265,849-row table.
    missing_pairs: dict = {}
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # ---- calibration sample -------------------------------------------
        # Thresholds must be calibrated on the WHOLE catalogue, but the pass
        # runs one partition at a time, so the calibration inputs are sampled
        # here and consumed in PHASE 3. Systematic sampling (every Nth row) so
        # the sample is not biased toward any partition's date range.
        _need = [c for c in CALIB_INPUT_COLUMNS if c in joined.columns]
        if _need:
            _step = max(1, len(joined) // CALIB_SAMPLE_PER_PARTITION)
            calib_parts.append(joined.iloc[::_step][_need + ["event_id"]])

        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        # Count DISTINCT events blocked by each input. Summing the per-string
        # Counter double-counts any input noted by two rules, which reported
        # 14,531,698 "events blocked" for a 7,265,849-event table.
        for event_id, entry in zip(joined["event_id"],
                                   label_out["missing_inputs"]):
            if entry:
                for nm in entry.split(","):
                    missing_pairs.setdefault(nm, set()).add(event_id)
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        # `y` is written in PHASE 3, after calibration.
        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "missing_label_inputs"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "y": y_writers["y"].rows_written if "y" in y_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  y {assembled['y']:,}  "
          f"meta {assembled['meta']:,}")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # =================================================================
    # PHASE 3 -- calibrate the weak thresholds, then label
    #
    # Calibration needs the whole distribution, but the pass is partitioned, so
    # it happens between PHASE 2 and the label pass. Without this the
    # persistence thresholds stayed tied to the pre-fix scale: INDUSTRIAL_FLARE
    # fell 3,571 -> 0 and MINING 73,060 -> 61 in one re-run.
    # =================================================================
    print()
    print("=" * 74)
    print("PHASE 3 -- CALIBRATE THRESHOLDS, THEN LABEL")
    print("=" * 74)

    calib_sample = (
        pd.concat(calib_parts, ignore_index=True)
        if calib_parts
        else pd.DataFrame()
    )
    print(f"  calibration sample: {len(calib_sample):,} rows")

    print()
    print("  FRP_PERSISTENCE distribution (the input that moved scale):")
    print(rc.distribution(
        calib_sample, ["FRP_PERSISTENCE"]).to_string(index=False))

    thresholds, evidence = rc.calibrate_to_target_counts(
        calib_sample, CALIB_TARGETS, base=weak.WIDE)
    calib_table, calib_report, _ = rc.calibration_report(
        calib_sample, thresholds, evidence)
    print()
    print("  calibrated thresholds:")
    print(calib_table.to_string(index=False))
    print()
    print("  weak labels on the SAMPLE, before vs after calibration:")
    print(calib_report.to_string(index=False))

    (QC_DIR / "p6_rule_calibration.json").write_text(json.dumps(
        {"targets": CALIB_TARGETS,
         "thresholds": rc.thresholds_to_dict(thresholds),
         "evidence": [asdict(e) for e in evidence]},
        indent=2, default=str))
    print(f"  Wrote {QC_DIR}/p6_rule_calibration.json")

    # Label every partition with the calibrated thresholds.
    print()
    print("  labelling all partitions with the calibrated thresholds ...")
    y_writer = None
    labelled = 0
    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue
        j = (
            pd.read_parquet(base_p)
            .merge(pd.read_parquet(t1_p), on="event_id", how="inner")
            .merge(pd.read_parquet(in_p), on="event_id", how="inner")
        )
        if j.empty:
            continue
        lo = lab.assign_labels(j)
        wo = weak.assign_weak_labels(j, thresholds)
        for cls, n in Counter(wo["label"]).items():
            weak_counts[cls] += n
        yf = pd.DataFrame({
            "event_id": j["event_id"].to_numpy(),
            "label": lo["label"],
            "label_rule": lo["rule"],
            "label_confidence": lo["confidence"],
            "sample_weight": lab.label_weights(lo["label"], lo["confidence"]),
            "label_weak": wo["label"],
            "label_weak_rule": wo["rule"],
            "label_weak_source": wo["label_source"],
            "missing_label_inputs": lo["missing_inputs"],
        })
        if y_writer is None:
            y_writer = cu.LazyParquetWriter(OUT_DIR / "y.parquet")
        y_writer.write_frame(yf)
        labelled += len(yf)
        if part % 5 == 0:
            print(f"    partition {part:>2d}  labelled={labelled:>9,}")

    if y_writer is not None:
        y_writer.close()
    print(f"  y.parquet rows: {labelled:,}")
    if labelled != assembled["X"]:
        print(f"  !! y has {labelled:,} rows, expected "
              f"{assembled['X']:,} -- the label pass lost rows")

    # ---- phase 4: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS, CALIBRATED thresholds (industrial-context")
    print("  hierarchy; manufactured from features the model sees, so")
    print("  never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_pairs:
        for nm in sorted(missing_pairs, key=lambda k: -len(missing_pairs[k])):
            n = len(missing_pairs[nm])
            print(f"    {nm:32s} {n:>10,} of {total:,} events blocked "
                  f"({100.0 * n / total:.1f}%)")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: len(v) for k, v in missing_pairs.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_thresholds": rc.thresholds_to_dict(thresholds),
        "weak_calibration_targets": CALIB_TARGETS,
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

In [1]:
# ============================================================================
# P6 -- COLAB CELL: assemble the training table
#
# Joins the two completed feature blocks onto the event base and builds
# X (52 features), y (labels) and meta (ids, time, split keys).
#
# Enforces the brief's requirements rather than leaving them to discipline:
#   * raw coordinates never enter X
#   * a cloud-blocked observation stays a gap (NaN), never a zero
#   * anomaly is judged against the site's own baseline
#
# Runs in chunks: 7.27M events never fit in memory alongside three frames.
# ============================================================================

import gc
import json
import time
from dataclasses import asdict
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import cluster_tier as ct
import colab_utils as cu
import dataset_builder as db
import feature_gap as fg
import feature_ledger as fl
import fire_labels as lab
import weak_labels as weak
import industrial_features as ind
import rule_calibration as rc
import training_design as td

INSPECT_ONLY = False
SITE_DEG = db.SITE_DEG
# 16 partitions -> ~454k events per step. Peak memory is one partition plus the
# current source chunk, not the whole catalogue. The first attempt held both
# feature blocks AND every accumulated output chunk (~10 GB) and the runtime
# died at batch 12 of 15.
N_PARTITIONS = db.N_PARTITIONS

# Which radius reading of the weak hierarchy to apply. Reported both ways in the
# QC so the choice is made on the counts, not on assertion.
#: Target class sizes used to calibrate the persistence thresholds. Chosen so a
#: class stays rare but learnable: ~0.05% of 7.27M is about 3,600 events.
CALIB_TARGETS = {
    weak.INDUSTRIAL_FLARE: 3_600,
    weak.INDUSTRIAL_FIRE: 400_000,
    weak.MINING: 60_000,
}

#: Rows sampled per partition for calibration. 16 x 12,500 = 200,000, ample
#: for percentiles in the 99.5th to 99.95th range.
CALIB_SAMPLE_PER_PARTITION = 12_500

#: Columns the weak rules read, plus the derived inputs they need.
CALIB_INPUT_COLUMNS = (
    "DIST_REFINERY_LOG", "DIST_NEAREST_FACILITY_LOG", "FACILITY_COUNT_5KM",
    "DIST_POWERPLANT_LOG", "DIST_PETROCHEMICAL_LOG", "DIST_STEEL_LOG",
    "DIST_CEMENT_LOG", "DIST_MINE_LOG", "FRP_PERSISTENCE",
    "DETECTION_COUNT_LOG", "TIME_TO_PEAK", "AREA_GROWTH_RATE",
    "SPREAD_SPEED_MEDIAN", "DIRECTION_CONSISTENCY", "INSIDE_FACILITY",
    "KNOWN_FLARE",
)

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}   tier1 {ct.N_CLUSTER_TIER}   "
      f"industrial {ind.N_INDUSTRIAL}   "
      f"labels {len(lab.LABEL_CLASSES)}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
_CANON = PROJECT_ROOT / "full_2012_2026"
FEATURES_DIR = _CANON / "features"
EVENTS_DIR = cu.pick_by_file_count(PROJECT_ROOT, "events")[0]
QC_DIR = PROJECT_ROOT / "qc"
OUT_DIR = FEATURES_DIR / "dataset"

TIER1_FILES = sorted((FEATURES_DIR / "lineage").glob("lineage_tier1_chunk_*.parquet"))
IND_FILES = sorted((FEATURES_DIR / "industrial").glob("industrial_chunk_*.parquet"))

print()
print("RESOLVED INPUTS")
print("-" * 74)
print(f"  tier1 chunks     : {len(TIER1_FILES)}")
print(f"  industrial chunks: {len(IND_FILES)}")
print(f"  events dir       : {EVENTS_DIR}")
print(f"  output           : {OUT_DIR}")
print("-" * 74)

if not TIER1_FILES or not IND_FILES:
    raise FileNotFoundError(
        "Both feature blocks must exist before P6.\n"
        f"  tier1 chunks found     : {len(TIER1_FILES)}\n"
        f"  industrial chunks found: {len(IND_FILES)}\n"
        "Run colab_p4a_tier1.py and colab_p3a_industrial.py first."
    )

# ---------------------------------------------------------------------------
# The feature list is fixed here, from the ledger, not discovered from the
# files -- so a chunk that silently gained or lost a column cannot change X.
# ---------------------------------------------------------------------------

TIER1_EXPECTED = list(ct.CLUSTER_TIER_FEATURES)
INDUSTRIAL_EXPECTED = list(ind.INDUSTRIAL_FEATURES)
FEATURE_COLUMNS = TIER1_EXPECTED + INDUSTRIAL_EXPECTED

# FRP_ANOMALY_VS_SITE and the site baseline are added by this cell.
DERIVED_FEATURE_COLUMNS = [
    "SITE_FRP_BASELINE", "SITE_N_EVENTS", "FRP_ANOMALY_VS_SITE",
    "MISSING_FEATURE_FRACTION",
    # Regime flags. 19 tier-1 columns are >90% NaN because 96.7% of events
    # are single-cluster and have no trend or geometry to measure. Filling them
    # would invent measurements, but the model still needs to know WHICH REGIME
    # an event is in. These flags give it that explicitly instead of leaving it
    # to infer regime from a pattern of NaNs.
    "N_DETECTIONS", "IS_SINGLE_DETECTION", "N_DETECTIONS_GE3",
    "PERSISTENCE_IS_VACUOUS",
]

ALL_FEATURES = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

# Guard before anything is written: no coordinate may reach the model.
db.assert_no_location_features(ALL_FEATURES)
print()
print(f"feature matrix: {len(ALL_FEATURES)} columns "
      f"({len(FEATURE_COLUMNS)} computed + {len(DERIVED_FEATURE_COLUMNS)} derived)")
print("location check: PASS -- no coordinate column in X")

# Pre-flight memory estimate, so a crash is explained rather than mysterious.
_est_total = int(
    pq.ParquetFile(TIER1_FILES[0]).metadata.num_rows
) * len(TIER1_FILES)
_partition_rows = _est_total / float(N_PARTITIONS)
print()
print("MEMORY PLAN")
print("-" * 74)
print(f"  whole-catalogue X would need "
      f"{db.human_bytes(db.partition_bytes_estimate(_est_total, len(ALL_FEATURES)))}"
      "  -> not attempted")
print(f"  {N_PARTITIONS} partitions, ~{_partition_rows:,.0f} events each")
print(f"  per-partition working set ~"
      f"{db.human_bytes(db.partition_bytes_estimate(_partition_rows, len(ALL_FEATURES) * 2))}")
print("-" * 74)

if INSPECT_ONLY:
    print()
    print("--- tier1 chunk schema (first file) ---")
    print(pq.ParquetFile(TIER1_FILES[0]).schema_arrow)
    print("--- industrial chunk schema (first file) ---")
    print(pq.ParquetFile(IND_FILES[0]).schema_arrow)
    print()
    print("INSPECTION COMPLETE -- nothing written.")
    print("Set INSPECT_ONLY = False to assemble the dataset.")
    _STOP = True
else:
    _STOP = False


# ===========================================================================
def run_assembly():
    """Assemble X / y / meta by streaming, one partition at a time.

    Three phases, none of which holds more than one chunk plus one partition:

      1. repartition the event base, tier 1 and industrial into N parquet
         shards keyed on a stable CRC32 of event_id
      2. for each partition: read its three shards, derive, label, and append
         straight to the output parquet writers
      3. delete the shards

    Nothing accumulates in a Python list. The previous version held both full
    feature blocks and every output chunk, needing >10 GB and dying at batch 12.
    """
    import shutil

    t0 = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    QC_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR = OUT_DIR / "_shards"
    if cu.path_ok(SHARD_DIR):
        shutil.rmtree(SHARD_DIR)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)

    base_path = next(
        (EVENTS_DIR / n for n in ("canonical_event_base.parquet",
                                   "event_base.parquet")
         if cu.path_ok(EVENTS_DIR / n)),
        None,
    )
    if base_path is None:
        raise FileNotFoundError(f"no event base parquet under {EVENTS_DIR}")
    print()
    print(f"event base: {base_path.name}")

    # geometry_n_satellites is required by the flare rule and lives ONLY in the
    # event base -- tier 1 does not carry it. Omitting it made every event
    # abstain, which is why the first assembly labelled 100% UNKNOWN.
    base_cols = [c for c in ("event_id", "event_centroid_lon",
                             "event_centroid_lat", "start_time",
                             "geometry_n_satellites")
                 if c in pq.ParquetFile(base_path).schema_arrow.names]
    if "geometry_n_satellites" not in base_cols:
        print("  !! geometry_n_satellites absent; the flare rule will abstain")

    # ---- phase 1: repartition every input into shards --------------------
    print()
    print("PHASE 1 -- repartition into shards")
    print("-" * 74)

    writers: dict = {}
    counts = Counter()

    def shard_writer(kind, part):
        key = (kind, part)
        if key not in writers:
            # LazyParquetWriter, not pq.ParquetWriter: the latter needs a schema
            # at construction, which is impossible for a shard that may never
            # receive a row.
            writers[key] = cu.LazyParquetWriter(
                SHARD_DIR / f"{kind}_p{part:02d}.parquet"
            )
        return writers[key]

    def write_shard(kind, frame):
        if frame.empty:
            return
        parts = db.partition_of(frame["event_id"].tolist(), N_PARTITIONS)
        for part in range(N_PARTITIONS):
            piece = frame[parts == part]
            if piece.empty:
                continue
            shard_writer(kind, part).write_frame(piece)
            counts[f"{kind}_p{part:02d}"] += len(piece)

    reader = pq.ParquetFile(base_path)
    n_base = 0
    for batch in reader.iter_batches(batch_size=500_000, columns=base_cols):
        frame = batch.to_pandas()
        n_base += len(frame)
        write_shard("base", frame)
        del frame
    print(f"  event base   : {n_base:,} rows -> {N_PARTITIONS} shards")

    for label, files in (("tier1", TIER1_FILES), ("industrial", IND_FILES)):
        n_rows = 0
        for path in files:
            frame = pd.read_parquet(path)
            n_rows += len(frame)
            write_shard(label, frame)
            del frame                      # released before the next file
        print(f"  {label:12s}: {n_rows:,} rows -> {N_PARTITIONS} shards")
        time.sleep(0)                      # yield, keeps the GC prompt

    for writer in writers.values():
        writer.close()
    # Drop shard writers whose partition never received a row: they would leave
    # empty files behind that the read phase would then try to open.
    # `wrote_anything`, NOT `opened`: close() above cleared the live writer, so
    # `opened` reads False for every shard and this deleted all sixteen. That
    # produced an empty PHASE 2 and a silent 0-event "success".
    empty = [
        f"{k[0]}_p{k[1]:02d}.parquet"
        for k, w in writers.items()
        if not w.wrote_anything
    ]
    if empty:
        print(f"  removed {len(empty)} empty shard(s)")
    for name in empty:
        (SHARD_DIR / name).unlink(missing_ok=True)
    writers.clear()
    shard_files = sorted(SHARD_DIR.glob("*.parquet"))
    shard_rows = sum(
        pq.ParquetFile(p).metadata.num_rows for p in shard_files
    )
    print(f"  shard dir: {SHARD_DIR}")
    print(f"  shards on disk: {len(shard_files)}  rows: {shard_rows:,}")
    if shard_rows < n_base:
        raise RuntimeError(
            f"PHASE 1 wrote {shard_rows:,} of {n_base:,} event-base rows. "
            "Refusing to continue: PHASE 2 would join against incomplete "
            "shards and report a smaller dataset as if it were complete."
        )

    # ---- phase 2: one partition at a time --------------------------------
    print()
    print("PHASE 2 -- build X / y / meta, one partition at a time")
    print("-" * 74)

    x_writers, y_writers, m_writers = {}, {}, {}
    calib_parts = []
    null_counts = {c: 0 for c in ALL_FEATURES}
    label_counts = Counter()
    # Distinct events blocked per missing input. A dict-of-sets, NOT a Counter:
    # summing per-string counts double-counted an input noted by two rules and
    # reported 14,531,698 blocked for a 7,265,849-row table.
    missing_pairs: dict = {}
    rule_counts = Counter()
    weak_counts = Counter()
    written = 0

    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue

        base_f = pd.read_parquet(base_p)
        t1_f = pd.read_parquet(t1_p)
        in_f = pd.read_parquet(in_p)

        joined = base_f.merge(t1_f, on="event_id", how="inner").merge(
            in_f, on="event_id", how="inner"
        )
        del base_f, t1_f, in_f
        if joined.empty:
            continue

        # derived
        flare_inputs = db.derive_flare_inputs(joined)
        # n_satellites comes straight from the event base.
        if "geometry_n_satellites" in joined.columns:
            flare_inputs["n_satellites"] = pd.to_numeric(
                joined["geometry_n_satellites"], errors="coerce"
            )
        # area_km2 is exactly invertible from the density: density = detections
        # per km2, and both inputs are present.
        joined = db.add_area_km2(joined)
        if "area_km2" in joined.columns:
            flare_inputs["area_km2"] = joined["area_km2"]
        joined = db.add_site_baseline(joined, site_deg=SITE_DEG)
        joined["FRP_ANOMALY_LOG"] = db.log_anomaly(joined["FRP_ANOMALY_VS_SITE"])
        joined["MISSING_FEATURE_FRACTION"] = db.missing_feature_fraction(
            joined, FEATURE_COLUMNS
        )
        joined = fg.add_single_detection_flags(joined)

        # labels
        label_out = lab.assign_labels(
            joined.assign(**flare_inputs.to_dict("series"))
        )
        # ---- calibration sample -------------------------------------------
        # Thresholds must be calibrated on the WHOLE catalogue, but the pass
        # runs one partition at a time, so the calibration inputs are sampled
        # here and consumed in PHASE 3. Systematic sampling (every Nth row) so
        # the sample is not biased toward any partition's date range.
        _need = [c for c in CALIB_INPUT_COLUMNS if c in joined.columns]
        if _need:
            _step = max(1, len(joined) // CALIB_SAMPLE_PER_PARTITION)
            calib_parts.append(joined.iloc[::_step][_need + ["event_id"]])

        # Two labellers, kept as separate columns.
        #   label_evidence : the strict, evidence-gated engine -- almost all
        #                    UNKNOWN, and defensible where it fires.
        #   label_weak     : the industrial-context rule hierarchy, which
        #                    labels far more but is manufactured from features
        #                    the model sees, so it is WEAK and carries its own
        #                    provenance. Never used to score model B.
        joined["label"] = label_out["label"]
        joined["label_rule"] = label_out["rule"]
        # Count DISTINCT events blocked by each input. Summing the per-string
        # Counter double-counts any input noted by two rules, which reported
        # 14,531,698 "events blocked" for a 7,265,849-event table.
        for event_id, entry in zip(joined["event_id"],
                                   label_out["missing_inputs"]):
            if entry:
                for nm in entry.split(","):
                    missing_pairs.setdefault(nm, set()).add(event_id)
        joined["label_confidence"] = label_out["confidence"]
        joined["sample_weight"] = lab.label_weights(
            label_out["label"], label_out["confidence"]
        )
        joined["missing_label_inputs"] = label_out["missing_inputs"]
        joined = td.add_split_keys(joined)

        for cls, n in Counter(label_out["label"]).items():
            label_counts[cls] += n
        for rule_name, n in Counter(label_out["rule"]).items():
            rule_counts[rule_name] += n
        for col in ALL_FEATURES:
            null_counts[col] += int(joined[col].isna().sum())

        # strip anything not in the declared feature list, then assert
        keep = [c for c in joined.columns if c in ALL_FEATURES]
        x_frame = joined[ALL_FEATURES]
        db.assert_no_location_features(list(x_frame.columns))

        # `y` is written in PHASE 3, after calibration.
        y_frame = joined[["event_id", "label", "label_rule",
                          "label_confidence", "sample_weight",
                          "missing_label_inputs"]]
        m_frame = joined[["event_id", "event_centroid_lon",
                          "event_centroid_lat", "start_time", "start_year",
                          "spatial_block", "site_id"]]
        del joined, keep

        for kind, frame, store in (
            ("X", x_frame, x_writers),
            ("meta", m_frame, m_writers),
        ):
            if kind not in store:
                store[kind] = cu.LazyParquetWriter(OUT_DIR / f"{kind}.parquet")
            store[kind].write_frame(frame)
            del frame

        written += len(x_frame)
        del x_frame, y_frame, m_frame, label_out, flare_inputs
        gc.collect()

        print(f"  partition {part:>2d}  rows={written:>9,}  "
              f"{time.time() - t0:>7.1f}s")

    for store in (x_writers, y_writers, m_writers):
        for writer in store.values():
            writer.close()
    # `y` is deliberately absent here: it is written in PHASE 3, after the
    # thresholds are calibrated. Reporting it as 0 at this point raised a false
    # "a write was lost" alarm on a completely successful run.
    assembled = {
        "X": x_writers["X"].rows_written if "X" in x_writers else 0,
        "meta": m_writers["meta"].rows_written if "meta" in m_writers else 0,
    }
    print()
    print(f"  written: X {assembled['X']:,}  meta {assembled['meta']:,}  "
          f"(y deferred to PHASE 3)")

    # A 0-event assembly must abort loudly. Reporting "NEXT: colab_p7_train.py"
    # after writing empty parquet files wastes a whole cycle and reads as
    # success.
    if assembled["X"] == 0:
        raise RuntimeError(
            "PHASE 2 assembled 0 events. The shards exist but the "
            "base/tier1/industrial join matched nothing. Check that the event "
            "id values are identical across all three inputs -- this ran to "
            "completion and printed NEXT, which is exactly the failure mode "
            "that must not pass silently."
        )
    for kind, n in assembled.items():
        if n != assembled["X"]:
            print(f"  !! {kind}.parquet has {n:,} rows, expected "
                  f"{assembled['X']:,} -- a write was lost")
    if assembled["X"] != written:
        print(f"  !! {written:,} events were processed but only "
              f"{assembled['X']:,} reached X.parquet")

    # =================================================================
    # PHASE 3 -- calibrate the weak thresholds, then label
    #
    # Calibration needs the whole distribution, but the pass is partitioned, so
    # it happens between PHASE 2 and the label pass. Without this the
    # persistence thresholds stayed tied to the pre-fix scale: INDUSTRIAL_FLARE
    # fell 3,571 -> 0 and MINING 73,060 -> 61 in one re-run.
    # =================================================================
    print()
    print("=" * 74)
    print("PHASE 3 -- CALIBRATE THRESHOLDS, THEN LABEL")
    print("=" * 74)

    calib_sample = (
        pd.concat(calib_parts, ignore_index=True)
        if calib_parts
        else pd.DataFrame()
    )
    print(f"  calibration sample: {len(calib_sample):,} rows")

    print()
    print("  FRP_PERSISTENCE distribution (the input that moved scale):")
    print(rc.distribution(
        calib_sample, ["FRP_PERSISTENCE"]).to_string(index=False))

    thresholds, evidence = rc.calibrate_to_target_counts(
        calib_sample, CALIB_TARGETS, base=weak.WIDE)
    calib_table, calib_report, _ = rc.calibration_report(
        calib_sample, thresholds, evidence)
    print()
    print("  calibrated thresholds:")
    print(calib_table.to_string(index=False))
    print()
    print("  weak labels on the SAMPLE, before vs after calibration:")
    print(calib_report.to_string(index=False))

    (QC_DIR / "p6_rule_calibration.json").write_text(json.dumps(
        {"targets": CALIB_TARGETS,
         "thresholds": rc.thresholds_to_dict(thresholds),
         "evidence": [asdict(e) for e in evidence]},
        indent=2, default=str))
    print(f"  Wrote {QC_DIR}/p6_rule_calibration.json")

    # Label every partition with the calibrated thresholds.
    print()
    print("  labelling all partitions with the calibrated thresholds ...")
    y_writer = None
    labelled = 0
    for part in range(N_PARTITIONS):
        base_p = SHARD_DIR / f"base_p{part:02d}.parquet"
        t1_p = SHARD_DIR / f"tier1_p{part:02d}.parquet"
        in_p = SHARD_DIR / f"industrial_p{part:02d}.parquet"
        if not (cu.path_ok(base_p) and cu.path_ok(t1_p) and cu.path_ok(in_p)):
            continue
        j = (
            pd.read_parquet(base_p)
            .merge(pd.read_parquet(t1_p), on="event_id", how="inner")
            .merge(pd.read_parquet(in_p), on="event_id", how="inner")
        )
        if j.empty:
            continue
        lo = lab.assign_labels(j)
        wo = weak.assign_weak_labels(j, thresholds)
        for cls, n in Counter(wo["label"]).items():
            weak_counts[cls] += n
        yf = pd.DataFrame({
            "event_id": j["event_id"].to_numpy(),
            "label": lo["label"],
            "label_rule": lo["rule"],
            "label_confidence": lo["confidence"],
            "sample_weight": lab.label_weights(lo["label"], lo["confidence"]),
            "label_weak": wo["label"],
            "label_weak_rule": wo["rule"],
            "label_weak_source": wo["label_source"],
            "missing_label_inputs": lo["missing_inputs"],
        })
        if y_writer is None:
            y_writer = cu.LazyParquetWriter(OUT_DIR / "y.parquet")
        y_writer.write_frame(yf)
        labelled += len(yf)
        if part % 5 == 0:
            print(f"    partition {part:>2d}  labelled={labelled:>9,}")

    if y_writer is not None:
        y_writer.close()
    print(f"  y.parquet rows: {labelled:,}")
    if labelled != assembled["X"]:
        print(f"  !! y has {labelled:,} rows, expected "
              f"{assembled['X']:,} -- the label pass lost rows")

    # ---- phase 4: drop the shards ---------------------------------------
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
    print()
    print(f"  shards removed; assembled {written:,} events")

    # ---- QC --------------------------------------------------------------
    print()
    print("=" * 74)
    print("P6 QC")
    print("=" * 74)
    total = max(written, 1)
    print(f"  events assembled : {written:,}  (event base had {n_base:,})")
    if written != n_base:
        print(f"  !! {n_base - written:,} events lost in the join")
    print()
    print("  labels")
    for cls in lab.LABEL_CLASSES:
        n = label_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    _unknown = label_counts.get(lab.UNKNOWN, 0)
    print(f"    {'supervised':20s} {total - _unknown:>10,}  "
          f"{100.0 * (total - _unknown) / total:>6.2f}%")
    print()
    print("  firing rules")
    for rule_name, n in sorted(rule_counts.items(), key=lambda kv: -kv[1]):
        print(f"    {rule_name:28s} {n:>10,}")
    print()
    print()
    print("  WEAK LABELS, CALIBRATED thresholds (industrial-context")
    print("  hierarchy; manufactured from features the model sees, so")
    print("  never used to score model B)")
    for cls in weak.WEAK_CLASSES:
        n = weak_counts.get(cls, 0)
        print(f"    {cls:20s} {n:>10,}  {100.0 * n / total:>6.2f}%")
    print()
    print("  WHY LABELS ABSTAIN -- rule inputs that were never available")
    print("  (this is the list to implement; each line is one fix)")
    if missing_pairs:
        for nm in sorted(missing_pairs, key=lambda k: -len(missing_pairs[k])):
            n = len(missing_pairs[nm])
            print(f"    {nm:32s} {n:>10,} of {total:,} events blocked "
                  f"({100.0 * n / total:.1f}%)")
    else:
        print("    (none -- every rule input was available)")
    print()
    print("  feature NaN%")
    for col in ALL_FEATURES:
        print(f"    {col:32s} {100.0 * null_counts[col] / total:>6.2f}%")

    dense = sum(1 for c in ALL_FEATURES
                if null_counts[c] / total < 0.5)
    print()
    print(f"  features >50% populated : {dense} of {len(ALL_FEATURES)}")

    manifest = {
        "n_events": int(written),
        "n_event_base": int(n_base),
        "n_partitions": N_PARTITIONS,
        "features": list(ALL_FEATURES),
        "label_counts": {c: int(label_counts.get(c, 0))
                         for c in lab.LABEL_CLASSES},
        "label_rules": {k: int(v) for k, v in rule_counts.items()},
        "label_missing_inputs": {k: len(v) for k, v in missing_pairs.items()},
        "weak_label_counts": {c: int(weak_counts.get(c, 0))
                              for c in weak.WEAK_CLASSES},
        "weak_label_inputs": list(weak.LABEL_INPUT_FEATURES),
        "weak_thresholds": rc.thresholds_to_dict(thresholds),
        "weak_calibration_targets": CALIB_TARGETS,
        "nan_fraction": {c: null_counts[c] / total for c in ALL_FEATURES},
        "review_confidence": db.REVIEW_CONFIDENCE,
        "alert_score_threshold": db.ALERT_SCORE_THRESHOLD,
        "coordinate_features": [],
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print()
    print(f"Wrote {OUT_DIR}/X.parquet, y.parquet, meta.parquet, manifest.json")
    print("NEXT: colab_p7_train.py")


if not _STOP:
    run_assembly()

ledger 1.3.0   tier1 34   industrial 18   labels 6
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------

RESOLVED INPUTS
--------------------------------------------------------------------------
  tier1 chunks   

KeyboardInterrupt: 

In [1]:
!pip install -q lightgbm


In [5]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td
import weak_labels as weak

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. LOAD -- sampled, float32
#
# The first attempt loaded all 7,265,849 rows as float64 and then handed the
# frame to LightGBM, which copies it into its own binary matrix: roughly 3.5 GB
# for the frame plus another 3.5 GB inside LightGBM. That exhausted the runtime.
#
# Three fixes:
#   * read only the columns actually used
#   * cast to float32 -- half the bytes, and LightGBM wants float32 anyway
#   * train on a stratified sample, reported explicitly rather than silently
# ---------------------------------------------------------------------------

MAX_TRAIN_ROWS = int(globals().get("MAX_TRAIN_ROWS", 1_500_000))
MAX_TEST_ROWS = int(globals().get("MAX_TEST_ROWS", 600_000))

t0 = time.time()
y = pd.read_parquet(DATASET_DIR / "y.parquet")
meta = pd.read_parquet(DATASET_DIR / "meta.parquet")
X = pd.read_parquet(DATASET_DIR / "X.parquet")
print(f"loaded X {X.shape}  y {y.shape}  meta {meta.shape}  "
      f"({time.time() - t0:.1f}s)")

# Hard gate: no coordinate may be in the model.
db.assert_no_location_features(list(X.columns))
print("location check: PASS")

# Which label column to train on.
#
#   "weak"    the industrial-context hierarchy in `weak_labels`. It labels a
#             fifth of events instead of none, but it is MANUFACTURED from
#             features the model also sees, so every metric on it is optimistic
#             by construction. Model B excludes every rule input.
#   "strict"  the evidence-gated engine. Almost entirely UNKNOWN, so it cannot
#             train yet -- it needs the land-cover and weather groups.
LABEL_COLUMN = "weak" if "label_weak" in y.columns else "strict"

labels = (y["label_weak"] if LABEL_COLUMN == "weak" else y["label"]).to_numpy()
weights = (y["sample_weight"].to_numpy() if "sample_weight" in y.columns
           else np.ones(len(labels), dtype="float64"))

# float32 halves the resident bytes and is LightGBM's native type. NaN is
# preserved, which is essential: NaN is how "could not measure" is encoded.
X = X.astype("float32")

TARGET = "INDUSTRIAL_FLARE"
print()
print(f"label column : {LABEL_COLUMN}")
if LABEL_COLUMN == "weak":
    print("  WEAK LABELS. These are manufactured from features the model sees,")
    print("  so PR-AUC here is an upper bound, NOT a generalisation estimate.")
    print("  Only an independent hand-labelled sample can give the latter.")
else:
    print("  strict labels -- nearly all UNKNOWN, so training cannot proceed.")
print()
print(f"target class : {TARGET}")
print("class counts (full catalogue):")
for cls in weak.WEAK_CLASSES:
    n = int((labels == cls).sum())
    print(f"  {cls:20s} {n:>10,}  {100.0 * n / max(len(labels), 1):>6.2f}%")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
split_kind = td.SPLIT_COMBINED
parts = td.make_split(meta, kind=split_kind, seed=0)
print(f"  split kind: {split_kind}")
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
print("  spatial leakage check: PASS (no shared block)")
if split_kind == td.SPLIT_OUT_OF_TIME:
    td.assert_no_time_leakage(meta[is_train], meta[is_test])
    print("  temporal leakage check: PASS (no test year in train)")
else:
    print("  temporal check: N/A -- the combined split trains on late events")
    print("    from non-held-out blocks by design; its guarantee is spatial.")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 2b. BOUND THE TRAINING SET
#
# LightGBM copies its input into a binary matrix, so a 7.27M x 60 float64 frame
# is roughly twice what the runtime has spare. Sampling is stratified on the
# TARGET so the minority class is not lost, and the sample size is printed so a
# metric is never mistaken for a full-catalogue result.
# ---------------------------------------------------------------------------

def stratified_sample(mask, cap, seed=0):
    """Indices from `mask`, at most `cap`, keeping the TARGET's prevalence."""
    idx = np.flatnonzero(mask)
    if len(idx) <= cap:
        return idx
    pos = idx[y_bin[idx] == 1]
    neg = idx[y_bin[idx] == 0]
    share = (len(pos) / len(idx)) if len(idx) else 0.0
    n_pos = min(len(pos), max(1, int(round(cap * share))))
    n_neg = min(len(neg), max(0, cap - n_pos))
    rng = np.random.default_rng(seed)
    take = np.concatenate([
        rng.choice(pos, n_pos, replace=False) if n_pos else np.empty(0, int),
        rng.choice(neg, n_neg, replace=False) if n_neg else np.empty(0, int),
    ])
    return np.sort(take)


train_idx = stratified_sample(is_train, MAX_TRAIN_ROWS, seed=0)
test_idx = stratified_sample(is_test, MAX_TEST_ROWS, seed=1)

print()
print("SAMPLE")
print(f"  train {len(train_idx):>9,} of {int(is_train.sum()):>10,}  "
      f"({100.0 * len(train_idx) / max(int(is_train.sum()), 1):.1f}%)")
print(f"  test  {len(test_idx):>9,} of {int(is_test.sum()):>10,}  "
      f"({100.0 * len(test_idx) / max(int(is_test.sum()), 1):.1f}%)")
print(f"  train positives {int(y_bin[train_idx].sum()):,}")
print(f"  test positives  {int(y_bin[test_idx].sum()):,}")
approx = (len(X.columns) * 4 * len(X)) / 1024 ** 3
print(f"  X as float32 ~{approx:.2f} GB, LightGBM holds a second copy")


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------

non_leak = [c for c in X.columns if c not in td.LABEL_DERIVED]
label_derived = [c for c in X.columns if c in td.LABEL_DERIVED]
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X.values[train_idx][:, [X.columns.get_loc(c) for c in non_leak]], y_bin[train_idx], weights[train_idx],
    X.values[test_idx][:, [X.columns.get_loc(c) for c in non_leak]], y_bin[test_idx], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[test_idx], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {X.shape[1]} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X.values[train_idx], y_bin[train_idx], weights[train_idx],
    X.values[test_idx], y_bin[test_idx], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[test_idx], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

dist = X["DIST_NEAREST_FACILITY_LOG"].to_numpy()
dist_m = np.where(np.isfinite(dist), np.power(10.0, dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    dist_m[train_idx], y_bin[train_idx]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
print("PER-CLASS METRICS (all classes, at the operating threshold)")
print("=" * 74)
for key in ("B", "A"):
    pred_cls = np.full(len(y_bin[test_idx]), weak.UNKNOWN, dtype=object)
    confident = prob_b if key == "B" else prob_a
    accepted = confident >= db.REVIEW_CONFIDENCE
    pred_cls[accepted] = TARGET
    report = per_class_report(
        labels[test_idx], pred_cls, list(weak.WEAK_CLASSES),
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "label_column": LABEL_COLUMN,
    "label_caveat": (
        "labels are weak: manufactured from features the model sees. "
        "PR-AUC here is an UPPER BOUND, not a generalisation estimate. "
        "An independent hand-labelled sample is required to evaluate."
    ) if LABEL_COLUMN == "weak" else "strict evidence-gated labels",
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(X.columns), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
print("Models are in memory only. Nothing has been written to Drive except")
print("the two QC files above.")

ledger 1.3.0
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------
dataset: /content/drive/MyDrive/Firm_Clusters/full_2012_2026/features/dataset
loaded X (7265849, 60)  y (7265849, 9)  meta (7265849, 7)  (17.1s)
lo

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
# ============================================================================
# BOOTSTRAP -- install every repo module, then verify them against each other.
#
# Self-healing by design: it installs EVERY .py it finds in the upload AND
# every .py already in Drive/pipeline. It has no MODULES list, so a module
# added to the repo later cannot be silently skipped by a stale pasted cell --
# which is exactly the failure that produced three rounds of
# `ModuleNotFoundError` in this project.
#
# Upload whatever .py files you have, then run this. It will tell you precisely
# which module is missing and where to get it, rather than dying on the first
# import.
# ============================================================================

import re
import sys
from pathlib import Path

from google.colab import files

PIPELINE = Path("/content/drive/MyDrive/Firm_Clusters/pipeline")
LOCAL = Path("/content")
DRIVE_OK = PIPELINE.exists()

#: Modules the cells import, with the minimum acceptable ledger version.
#: Every module the pipeline imports. A module missing from this list is not
#: reported until some later cell dies with a bare ImportError, which is how
#: cluster_tier / weak_labels / dataset_builder went missing twice.
REQUIRED = {
    "feature_ledger": None,      # version checked separately
    "industrial_features": None,
    "lineage_features": None,
    "cluster_tier": None,
    "fire_labels": None,
    "weak_labels": None,
    "dataset_builder": None,
    "training_design": None,
    "feature_gap": None,
    "colab_utils": None,
}

print("=" * 70)
print("BOOTSTRAP")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Collect candidates: the upload first, then whatever is already on Drive.
# ---------------------------------------------------------------------------

candidates = {}   # module name -> (source, bytes)

uploaded = files.upload()
print(f"upload dialog returned {len(uploaded)} file(s): {sorted(uploaded)}")

for key, payload in uploaded.items():
    if not key.endswith(".py"):
        continue
    # macOS/Windows append " (2)" to a duplicated filename, so strip any suffix
    # before matching rather than doing an exact key lookup.
    stem = re.sub(r"\s*\(\d+\)$", "", Path(key).stem)
    candidates[stem] = ("upload", payload)

if DRIVE_OK:
    for path in sorted(PIPELINE.glob("*.py")):
        stem = path.stem
        if stem in candidates:
            continue          # the upload is newer, keep it
        try:
            candidates[stem] = ("drive", path.read_bytes())
        except OSError as exc:
            print(f"  !! could not read {path}: {exc}")
else:
    print()
    print("  Drive pipeline dir not visible; relying on the upload alone.")

# ---------------------------------------------------------------------------
# 2. Install, reporting anything still missing.
# ---------------------------------------------------------------------------

PIPELINE.mkdir(parents=True, exist_ok=True)
installed, missing = [], []

for mod in REQUIRED:
    entry = candidates.get(mod)
    if entry is None:
        missing.append(mod)
        continue
    source, payload = entry
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    installed.append(mod)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes)")

# Anything else that showed up is installed too, so a future module works
# without editing this cell.
for mod, (source, payload) in sorted(candidates.items()):
    if mod in installed:
        continue
    (LOCAL / f"{mod}.py").write_bytes(payload)
    (PIPELINE / f"{mod}.py").write_bytes(payload)
    sys.modules.pop(mod, None)
    print(f"  {mod:22s} <- {source:6s} ({len(payload):,} bytes) [extra]")

print()
if missing:
    print("=" * 70)
    print(f"MISSING: {missing}")
    print("=" * 70)
    print("Re-run the file picker and select ALL of these together, from the")
    print("repo folder konsors-sih2:")
    for mod in missing:
        print(f"  - {mod}.py")
    print()
    print("The picker returns only what you highlight, not the whole folder.")
    raise SystemExit(1)

print(f"installed: {installed}")

# ---------------------------------------------------------------------------
# 3. Verify. Import failures here name the module instead of crashing a
#    feature cell three steps later.
# ---------------------------------------------------------------------------

print()
missing_import = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except Exception as exc:                      # noqa: BLE001
        missing_import.append(f"{mod} ({exc.__class__.__name__}: {exc})")

if missing_import:
    print("=" * 70)
    print("IMPORT FAILED")
    print("=" * 70)
    for line in missing_import:
        print(f"  {line}")
    print()
    print("A file installed but will not import is usually a syntax error or a")
    print("missing dependency. Re-upload that module from the repo.")
    raise SystemExit(1)

import colab_utils as cu
import cluster_tier as ct
import dataset_builder as db
import feature_gap as fgap
import feature_ledger as fl
import fire_labels as lab
import industrial_features as ind
import lineage_features as lf
import training_design as td
import weak_labels as weak


def version_parts(v):
    out = []
    for chunk in str(v).split("."):
        digits = "".join(c for c in chunk if c.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out)


def too_old(found, expected):
    if found is None:
        return True
    f, e = version_parts(found), version_parts(expected)
    w = max(len(f), len(e))
    f, e = f + (0,) * (w - len(f)), e + (0,) * (w - len(e))
    return f < e


print()
print("=" * 70)
print("CROSS-MODULE CHECKS")
print("=" * 70)

problems = []

if too_old(fl.LEDGER_VERSION, "1.3.0"):
    problems.append(
        f"stale feature_ledger: {fl.LEDGER_VERSION} < 1.3.0 -- re-upload it"
    )
if not hasattr(fl, "LOCAL_INPUT_HINTS"):
    problems.append("stale feature_ledger: no LOCAL_INPUT_HINTS")
if not hasattr(fl, "S_NEW_IMPL"):
    problems.append("stale feature_ledger: no NEW_IMPLEMENTATION status")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial feature order drifted from the ledger")
if ind.REQUIRED_CRS != "EPSG:6933":
    problems.append(f"unexpected CRS {ind.REQUIRED_CRS}")
if lf.N_LINEAGE != 54:
    problems.append(f"lineage block is {lf.N_LINEAGE}, expected 54")
if not set(lf.LINEAGE_FEATURES) <= set(fl.FEATURES_141):
    problems.append("lineage features are not a subset of the ledger")
if len(lab.LABEL_CLASSES) != 6:
    problems.append(f"{len(lab.LABEL_CLASSES)} label classes, expected 6")
if 107 not in cu.DEAD_MOUNT_ERRNOS:
    problems.append("colab_utils is missing the errno-107 dead-mount entry")
if ind.INDUSTRIAL_FEATURES != fl.FEATURE_GROUPS[fl.GROUP_INDUSTRIAL]:
    problems.append("industrial block does not match the ledger")
if ct.N_CLUSTER_TIER + len(weak.LABEL_INPUT_FEATURES) < 1:
    problems.append("cluster_tier / weak_labels look empty")
if len(fl.FEATURES_141) != 141:
    problems.append(f"ledger has {len(fl.FEATURES_141)} features, expected 141")
if not hasattr(db, "SITE_DEG") or not hasattr(fgap, "gap_report"):
    problems.append("dataset_builder / feature_gap are stale")

if problems:
    print("FAILED:")
    for line in problems:
        print(f"  - {line}")
    raise SystemExit(1)

print(f"  ledger      : {fl.LEDGER_VERSION}")
print(f"  industrial  : {ind.N_INDUSTRIAL} features")
print(f"  lineage     : {lf.N_LINEAGE} features")
print(f"  labels      : {len(lab.LABEL_CLASSES)} classes")
print(f"  weak labels : {len(weak.LABEL_INPUT_FEATURES)} rule inputs published")
print(f"  site grid   : {db.SITE_DEG} deg")
print("  all modules agree")
print()
print(cu.drive_report())
print()
print("BOOTSTRAP OK -- next: colab_p3a_industrial.py, colab_p4a_lineage.py")
print("=" * 70)

BOOTSTRAP


Saving rule_calibration.py to rule_calibration (3).py
Saving cluster_tier.py to cluster_tier (4).py
Saving colab_utils.py to colab_utils (5).py
Saving dataset_builder.py to dataset_builder (4).py
Saving feature_gap.py to feature_gap (4).py
Saving fire_labels.py to fire_labels (4).py
Saving lineage_features.py to lineage_features (4).py
Saving weak_labels.py to weak_labels (4).py
Saving training_design.py to training_design (4).py
Saving feature_ledger.py to feature_ledger (8).py
Saving industrial_features.py to industrial_features (5).py
upload dialog returned 11 file(s): ['cluster_tier (4).py', 'colab_utils (5).py', 'dataset_builder (4).py', 'feature_gap (4).py', 'feature_ledger (8).py', 'fire_labels (4).py', 'industrial_features (5).py', 'lineage_features (4).py', 'rule_calibration (3).py', 'training_design (4).py', 'weak_labels (4).py']
  feature_ledger         <- upload (31,302 bytes)
  industrial_features    <- upload (19,120 bytes)
  lineage_features       <- upload (34,759 bytes

In [ ]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td
import weak_labels as weak

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. LOAD -- sampled, float32
#
# The first attempt loaded all 7,265,849 rows as float64 and then handed the
# frame to LightGBM, which copies it into its own binary matrix: roughly 3.5 GB
# for the frame plus another 3.5 GB inside LightGBM. That exhausted the runtime.
#
# Three fixes:
#   * read only the columns actually used
#   * cast to float32 -- half the bytes, and LightGBM wants float32 anyway
#   * train on a stratified sample, reported explicitly rather than silently
# ---------------------------------------------------------------------------

MAX_TRAIN_ROWS = int(globals().get("MAX_TRAIN_ROWS", 1_500_000))
MAX_TEST_ROWS = int(globals().get("MAX_TEST_ROWS", 600_000))

t0 = time.time()
y = pd.read_parquet(DATASET_DIR / "y.parquet")
meta = pd.read_parquet(DATASET_DIR / "meta.parquet")
X = pd.read_parquet(DATASET_DIR / "X.parquet")
print(f"loaded X {X.shape}  y {y.shape}  meta {meta.shape}  "
      f"({time.time() - t0:.1f}s)")

# Hard gate: no coordinate may be in the model.
db.assert_no_location_features(list(X.columns))
print("location check: PASS")

# Which label column to train on.
#
#   "weak"    the industrial-context hierarchy in `weak_labels`. It labels a
#             fifth of events instead of none, but it is MANUFACTURED from
#             features the model also sees, so every metric on it is optimistic
#             by construction. Model B excludes every rule input.
#   "strict"  the evidence-gated engine. Almost entirely UNKNOWN, so it cannot
#             train yet -- it needs the land-cover and weather groups.
LABEL_COLUMN = "weak" if "label_weak" in y.columns else "strict"

labels = (y["label_weak"] if LABEL_COLUMN == "weak" else y["label"]).to_numpy()
weights = (y["sample_weight"].to_numpy() if "sample_weight" in y.columns
           else np.ones(len(labels), dtype="float64"))

# float32 halves the resident bytes and is LightGBM's native type. NaN is
# preserved, which is essential: NaN is how "could not measure" is encoded.
X = X.astype("float32")

# Which class to fit. INDUSTRIAL_FLARE has only 3,571 events in the whole
# catalogue -- 894 in the training sample and 33 in the test set -- so its
# metrics are dominated by sampling noise and both models early-stopped at
# iteration 3. INDUSTRIAL_FIRE has 1,119,254 and is the first target that can
# actually support a model. Change this to compare.
TARGET = globals().get("TARGET", "INDUSTRIAL_FIRE")
print()
print(f"label column : {LABEL_COLUMN}")
if LABEL_COLUMN == "weak":
    print("  WEAK LABELS. These are manufactured from features the model sees,")
    print("  so PR-AUC here is an upper bound, NOT a generalisation estimate.")
    print("  Only an independent hand-labelled sample can give the latter.")
else:
    print("  strict labels -- nearly all UNKNOWN, so training cannot proceed.")
print()
print(f"target class : {TARGET}")
print("class counts (full catalogue):")
for cls in weak.WEAK_CLASSES:
    n = int((labels == cls).sum())
    print(f"  {cls:20s} {n:>10,}  {100.0 * n / max(len(labels), 1):>6.2f}%")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
split_kind = td.SPLIT_COMBINED
parts = td.make_split(meta, kind=split_kind, seed=0)
print(f"  split kind: {split_kind}")
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
print("  spatial leakage check: PASS (no shared block)")
if split_kind == td.SPLIT_OUT_OF_TIME:
    td.assert_no_time_leakage(meta[is_train], meta[is_test])
    print("  temporal leakage check: PASS (no test year in train)")
else:
    print("  temporal check: N/A -- the combined split trains on late events")
    print("    from non-held-out blocks by design; its guarantee is spatial.")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 2b. BOUND THE TRAINING SET
#
# LightGBM copies its input into a binary matrix, so a 7.27M x 60 float64 frame
# is roughly twice what the runtime has spare. Sampling is stratified on the
# TARGET so the minority class is not lost, and the sample size is printed so a
# metric is never mistaken for a full-catalogue result.
# ---------------------------------------------------------------------------

def stratified_sample(mask, cap, seed=0):
    """Indices from `mask`, at most `cap`, keeping the TARGET's prevalence."""
    idx = np.flatnonzero(mask)
    if len(idx) <= cap:
        return idx
    pos = idx[y_bin[idx] == 1]
    neg = idx[y_bin[idx] == 0]
    share = (len(pos) / len(idx)) if len(idx) else 0.0
    n_pos = min(len(pos), max(1, int(round(cap * share))))
    n_neg = min(len(neg), max(0, cap - n_pos))
    rng = np.random.default_rng(seed)
    take = np.concatenate([
        rng.choice(pos, n_pos, replace=False) if n_pos else np.empty(0, int),
        rng.choice(neg, n_neg, replace=False) if n_neg else np.empty(0, int),
    ])
    return np.sort(take)


train_idx = stratified_sample(is_train, MAX_TRAIN_ROWS, seed=0)
test_idx = stratified_sample(is_test, MAX_TEST_ROWS, seed=1)

print()
print("SAMPLE")
print(f"  train {len(train_idx):>9,} of {int(is_train.sum()):>10,}  "
      f"({100.0 * len(train_idx) / max(int(is_train.sum()), 1):.1f}%)")
print(f"  test  {len(test_idx):>9,} of {int(is_test.sum()):>10,}  "
      f"({100.0 * len(test_idx) / max(int(is_test.sum()), 1):.1f}%)")
print(f"  train positives {int(y_bin[train_idx].sum()):,}")
print(f"  test positives  {int(y_bin[test_idx].sum()):,}")
approx = (len(X.columns) * 4 * len(X)) / 1024 ** 3
print(f"  X as float32 ~{approx:.2f} GB, LightGBM holds a second copy")


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------

non_leak = [c for c in X.columns if c not in td.LABEL_DERIVED]
label_derived = [c for c in X.columns if c in td.LABEL_DERIVED]
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X.values[train_idx][:, [X.columns.get_loc(c) for c in non_leak]], y_bin[train_idx], weights[train_idx],
    X.values[test_idx][:, [X.columns.get_loc(c) for c in non_leak]], y_bin[test_idx], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[test_idx], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {X.shape[1]} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X.values[train_idx], y_bin[train_idx], weights[train_idx],
    X.values[test_idx], y_bin[test_idx], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[test_idx], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

dist = X["DIST_NEAREST_FACILITY_LOG"].to_numpy()
dist_m = np.where(np.isfinite(dist), np.power(10.0, dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    dist_m[train_idx], y_bin[train_idx]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
print(f"PER-CLASS METRICS -- binary {TARGET} vs NOT_{TARGET},")
print("at the 0.60 review threshold. This model does NOT predict the other")
print("four classes; the full 6-class problem needs a multilabel head or six")
print("separate models, which is a later step.")
print("=" * 74)
for key in ("B", "A"):
    # This is a BINARY model (TARGET vs everything else), so reporting it in
    # six classes claimed that every non-TARGET row was predicted UNKNOWN --
    # which is a category it never saw. Only the target row and the
    # "everything else" row are actually being decided.
    pred_cls = np.full(len(y_bin[test_idx]), "NOT_" + TARGET, dtype=object)
    confident = prob_b if key == "B" else prob_a
    pred_cls[confident >= db.REVIEW_CONFIDENCE] = TARGET
    report = per_class_report(
        labels[test_idx], pred_cls, [TARGET, "NOT_" + TARGET],
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "label_column": LABEL_COLUMN,
    "label_caveat": (
        "labels are weak: manufactured from features the model sees. "
        "PR-AUC here is an UPPER BOUND, not a generalisation estimate. "
        "An independent hand-labelled sample is required to evaluate."
    ) if LABEL_COLUMN == "weak" else "strict evidence-gated labels",
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(X.columns), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
print("Models are in memory only. Nothing has been written to Drive except")
print("the two QC files above.")

ledger 1.3.0
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------
dataset: /content/drive/MyDrive/Firm_Clusters/full_2012_2026/features/dataset
loaded X (7265849, 60)  y (7265849, 9)  meta (7265849, 7)  (18.5s)
lo

In [3]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td
import weak_labels as weak

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. LOAD -- sampled, float32
#
# The first attempt loaded all 7,265,849 rows as float64 and then handed the
# frame to LightGBM, which copies it into its own binary matrix: roughly 3.5 GB
# for the frame plus another 3.5 GB inside LightGBM. That exhausted the runtime.
#
# Three fixes:
#   * read only the columns actually used
#   * cast to float32 -- half the bytes, and LightGBM wants float32 anyway
#   * train on a stratified sample, reported explicitly rather than silently
# ---------------------------------------------------------------------------

MAX_TRAIN_ROWS = int(globals().get("MAX_TRAIN_ROWS", 1_500_000))
MAX_TEST_ROWS = int(globals().get("MAX_TEST_ROWS", 600_000))

t0 = time.time()

# --- read ONLY what is needed, and stream X --------------------------------
# Two earlier versions exhausted the runtime. Both read the WHOLE table and
# only sampled afterwards, which is backwards: sampling cannot save memory that
# has already been spent. 7,265,849 x 60 is 1.6 GB as float32, and the four
# string columns in y plus three in meta roughly double that once pandas'
# object arrays are counted.
#
# So: read y and meta with the columns actually needed, decide the split and
# the sample from those small tables, then stream X one row group at a time
# keeping ONLY the sampled rows. Peak becomes the sample, not the catalogue.
t0 = time.time()

_y_cols = set(pq.ParquetFile(DATASET_DIR / "y.parquet").schema_arrow.names)
_meta_cols = set(
    pq.ParquetFile(DATASET_DIR / "meta.parquet").schema_arrow.names
)

y = pd.read_parquet(
    DATASET_DIR / "y.parquet",
    columns=[c for c in ("label_weak", "label", "sample_weight")
             if c in _y_cols],
)
meta = pd.read_parquet(
    DATASET_DIR / "meta.parquet",
    columns=[c for c in ("event_id", "event_centroid_lon", "event_centroid_lat",
                         "start_time", "start_year")
             if c in _meta_cols],
)
print(f"loaded y {y.shape}  meta {meta.shape}  ({time.time() - t0:.1f}s)")
print("  y columns kept    :", list(y.columns))
print("  meta columns kept :", list(meta.columns))

# Hard gate on the feature NAMES, straight from the parquet footer, so it runs
# before a single row is read.
_X_COLS = list(pq.ParquetFile(DATASET_DIR / "X.parquet").schema_arrow.names)
db.assert_no_location_features(_X_COLS)
print("location check: PASS")
print(f"  X has {len(_X_COLS)} columns (from the footer, not a read)")

# Which label column to train on.
#
#   "weak"    the industrial-context hierarchy in `weak_labels`. It labels a
#             fifth of events instead of none, but it is MANUFACTURED from
#             features the model also sees, so every metric on it is optimistic
#             by construction. Model B excludes every rule input.
#   "strict"  the evidence-gated engine. Almost entirely UNKNOWN, so it cannot
#             train yet -- it needs the land-cover and weather groups.
LABEL_COLUMN = "weak" if "label_weak" in y.columns else "strict"

labels = (y["label_weak"] if LABEL_COLUMN == "weak" else y["label"]).to_numpy()
weights = (y["sample_weight"].to_numpy() if "sample_weight" in y.columns
           else np.ones(len(labels), dtype="float64"))

# `X` does not exist yet. It is filled AFTER the sample is chosen, straight
# into preallocated float32 arrays, one row group at a time. NaN is preserved,
# which is essential: NaN is how "could not measure" is encoded.

# Which class to fit. INDUSTRIAL_FLARE has only 3,571 events in the whole
# catalogue -- 894 in the training sample and 33 in the test set -- so its
# metrics are dominated by sampling noise and both models early-stopped at
# iteration 3. INDUSTRIAL_FIRE has 1,119,254 and is the first target that can
# actually support a model. Change this to compare.
TARGET = globals().get("TARGET", "INDUSTRIAL_FIRE")
print()
print(f"label column : {LABEL_COLUMN}")
if LABEL_COLUMN == "weak":
    print("  WEAK LABELS. These are manufactured from features the model sees,")
    print("  so PR-AUC here is an upper bound, NOT a generalisation estimate.")
    print("  Only an independent hand-labelled sample can give the latter.")
else:
    print("  strict labels -- nearly all UNKNOWN, so training cannot proceed.")
print()
print(f"target class : {TARGET}")
print("class counts (full catalogue):")
for cls in weak.WEAK_CLASSES:
    n = int((labels == cls).sum())
    print(f"  {cls:20s} {n:>10,}  {100.0 * n / max(len(labels), 1):>6.2f}%")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
split_kind = td.SPLIT_COMBINED
parts = td.make_split(meta, kind=split_kind, seed=0)
print(f"  split kind: {split_kind}")
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
print("  spatial leakage check: PASS (no shared block)")
if split_kind == td.SPLIT_OUT_OF_TIME:
    td.assert_no_time_leakage(meta[is_train], meta[is_test])
    print("  temporal leakage check: PASS (no test year in train)")
else:
    print("  temporal check: N/A -- the combined split trains on late events")
    print("    from non-held-out blocks by design; its guarantee is spatial.")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 2b. BOUND THE TRAINING SET
#
# LightGBM copies its input into a binary matrix, so a 7.27M x 60 float64 frame
# is roughly twice what the runtime has spare. Sampling is stratified on the
# TARGET so the minority class is not lost, and the sample size is printed so a
# metric is never mistaken for a full-catalogue result.
# ---------------------------------------------------------------------------

def stratified_sample(mask, cap, seed=0):
    """Indices from `mask`, at most `cap`, keeping the TARGET's prevalence."""
    idx = np.flatnonzero(mask)
    if len(idx) <= cap:
        return idx
    pos = idx[y_bin[idx] == 1]
    neg = idx[y_bin[idx] == 0]
    share = (len(pos) / len(idx)) if len(idx) else 0.0
    n_pos = min(len(pos), max(1, int(round(cap * share))))
    n_neg = min(len(neg), max(0, cap - n_pos))
    rng = np.random.default_rng(seed)
    take = np.concatenate([
        rng.choice(pos, n_pos, replace=False) if n_pos else np.empty(0, int),
        rng.choice(neg, n_neg, replace=False) if n_neg else np.empty(0, int),
    ])
    return np.sort(take)


train_idx = stratified_sample(is_train, MAX_TRAIN_ROWS, seed=0)
test_idx = stratified_sample(is_test, MAX_TEST_ROWS, seed=1)

print()
print("SAMPLE")
total_rows = pq.ParquetFile(DATASET_DIR / "X.parquet").metadata.num_rows
print(f"  train {len(train_idx):>9,} of {int(is_train.sum()):>10,}  "
      f"({100.0 * len(train_idx) / max(int(is_train.sum()), 1):.1f}%)")
print(f"  test  {len(test_idx):>9,} of {int(is_test.sum()):>10,}  "
      f"({100.0 * len(test_idx) / max(int(is_test.sum()), 1):.1f}%)")
print(f"  train positives {int(y_bin[train_idx].sum()):,}")
print(f"  test positives  {int(y_bin[test_idx].sum()):,}")
_full_gb = len(_X_COLS) * 4 * total_rows / 1024 ** 3
print(f"  a full float32 frame would be {_full_gb:.2f} GB; streaming avoids it")



# --- the partition must be the CURRENT one -----------------------------------
# A stale training_design.py computed model B as NON_LEAK - industrial, which
# leaves six weak-rule inputs (FRP_PERSISTENCE, DETECTION_COUNT_LOG,
# TIME_TO_PEAK, AREA_GROWTH_RATE, SPREAD_SPEED_MEDIAN, DIRECTION_CONSISTENCY)
# in the "scientific" model -- and those are the INDUSTRIAL_FIRE rule's own
# shape branch. That run reported 42 features and PR-AUC 0.527, both of which
# were artefacts. Fail loudly instead.
_model_a_cols, _model_b_cols, _leak = td.split_feature_sets(_X_COLS)
_expected_b = len(_model_b_cols)
_still_leaking = sorted(set(_model_b_cols) & set(weak.LABEL_INPUT_FEATURES))
if _still_leaking:
    raise SystemExit(
        "STALE training_design.py or weak_labels.py: these weak-rule inputs "
        f"are in model B: {_still_leaking}.\n"
        "Re-upload training_design.py and weak_labels.py, then re-run."
    )
print(f"  partition check: model A {len(_model_a_cols)}  model B "
      f"{_expected_b}  (rule inputs excluded: {len(_leak['excluded_from_b'])})")

non_leak = list(_model_b_cols)
label_derived = [c for c in _X_COLS if c in td.LABEL_DERIVED]
# ---------------------------------------------------------------------------
# 3. STREAM X -- keep only the sampled rows
#
# The catalogue is walked one row group at a time and only the sampled
# positions are retained, so peak memory is the SAMPLE rather than 7.27M rows.
# A full 60-column float32 frame is 1.6 GB; the sample is about 0.4 GB, and
# LightGBM's copy of it is smaller again.
# ---------------------------------------------------------------------------

_all = np.concatenate([train_idx, test_idx])
_all_sorted = np.sort(_all)
_keep_rows = _all_sorted

print()
print("streaming X, keeping only the sampled rows ...")
X_sampled = np.empty((len(_keep_rows), len(_X_COLS)), dtype="float32")
_filled = 0
_t0 = time.time()
_pf = pq.ParquetFile(DATASET_DIR / "X.parquet")
pf_rows = _pf.metadata.num_rows
_offset = 0
for _batch in _pf.iter_batches(batch_size=250_000, columns=_X_COLS):
    _n = _batch.num_rows
    _start = _offset
    _end = _offset + _n
    _lo = np.searchsorted(_keep_rows, _start)
    _hi = np.searchsorted(_keep_rows, _end)
    if _hi > _lo:
        local = _keep_rows[_lo:_hi] - _start
        block = _batch.to_pandas().to_numpy(dtype="float32", copy=True)
        X_sampled[_lo:_hi] = block[local]
        _filled += _hi - _lo
        del block
    _offset = _end
    if _offset % 1_000_000 < _n:
        print(f"    {_offset:>10,} / {pf_rows:,} rows  "
              f"({_filled:,} kept, {time.time() - _t0:.0f}s)")

print(f"  filled {_filled:,} of {len(_keep_rows):,} requested")
if _filled != len(_keep_rows):
    raise SystemExit(
        f"streaming fill got {_filled:,} rows, expected {len(_keep_rows):,}. "
        "The sample indices and the parquet row order must agree."
    )

_train_local = np.searchsorted(_keep_rows, train_idx)
_test_local = np.searchsorted(_keep_rows, test_idx)
_b_cols = [_X_COLS.index(c) for c in _model_b_cols]
X_train_b = np.ascontiguousarray(X_sampled[_train_local][:, _b_cols])
X_test_b = np.ascontiguousarray(X_sampled[_test_local][:, _b_cols])
X_train_a = np.ascontiguousarray(X_sampled[_train_local])
X_test_a = np.ascontiguousarray(X_sampled[_test_local])
del X_sampled

_mb = X_train_b.nbytes / 1024 ** 3
_ma = X_train_a.nbytes / 1024 ** 3
print(f"  model A train matrix {_ma:.2f} GB   model B train matrix {_mb:.2f} GB")
print("  (the 7.27M-row catalogue was never resident)")


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X_train_b, y_bin[train_idx], weights[train_idx],
    X_test_b, y_bin[test_idx], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[test_idx], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {len(_model_a_cols)} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X_train_a, y_bin[train_idx], weights[train_idx],
    X_test_a, y_bin[test_idx], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[test_idx], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

# Distance is only needed for the baseline, and only for the sampled
# rows, so it is read from the matrices already in memory.
_dist_col = _X_COLS.index("DIST_NEAREST_FACILITY_LOG")
_dist = np.concatenate([X_train_a[:, _dist_col], X_test_a[:, _dist_col]])
dist_m = np.where(np.isfinite(_dist), np.power(10.0, _dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    _dist[:len(train_idx)], y_bin[train_idx]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
print(f"PER-CLASS METRICS -- binary {TARGET} vs NOT_{TARGET},")
print("at the 0.60 review threshold. This model does NOT predict the other")
print("four classes; the full 6-class problem needs a multilabel head or six")
print("separate models, which is a later step.")
print("=" * 74)
for key in ("B", "A"):
    # This is a BINARY model (TARGET vs everything else), so reporting it in
    # six classes claimed that every non-TARGET row was predicted UNKNOWN --
    # which is a category it never saw. Only the target row and the
    # "everything else" row are actually being decided.
    pred_cls = np.full(len(y_bin[test_idx]), "NOT_" + TARGET, dtype=object)
    confident = prob_b if key == "B" else prob_a
    pred_cls[confident >= db.REVIEW_CONFIDENCE] = TARGET
    report = per_class_report(
        labels[test_idx], pred_cls, [TARGET, "NOT_" + TARGET],
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "label_column": LABEL_COLUMN,
    "label_caveat": (
        "labels are weak: manufactured from features the model sees. "
        "PR-AUC here is an UPPER BOUND, not a generalisation estimate. "
        "An independent hand-labelled sample is required to evaluate."
    ) if LABEL_COLUMN == "weak" else "strict evidence-gated labels",
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(_model_a_cols), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
print("Models are in memory only. Nothing has been written to Drive except")
print("the two QC files above.")

ledger 1.3.0
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------
dataset: /content/drive/MyDrive/Firm_Clusters/full_2012_2026/features/dataset
loaded y (7265849, 3)  meta (7265849, 5)  (9.8s)
  y columns kept    

In [4]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td
import weak_labels as weak

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. LOAD -- sampled, float32
#
# The first attempt loaded all 7,265,849 rows as float64 and then handed the
# frame to LightGBM, which copies it into its own binary matrix: roughly 3.5 GB
# for the frame plus another 3.5 GB inside LightGBM. That exhausted the runtime.
#
# Three fixes:
#   * read only the columns actually used
#   * cast to float32 -- half the bytes, and LightGBM wants float32 anyway
#   * train on a stratified sample, reported explicitly rather than silently
# ---------------------------------------------------------------------------

MAX_TRAIN_ROWS = int(globals().get("MAX_TRAIN_ROWS", 1_500_000))
MAX_TEST_ROWS = int(globals().get("MAX_TEST_ROWS", 600_000))

t0 = time.time()

# --- read ONLY what is needed, and stream X --------------------------------
# Two earlier versions exhausted the runtime. Both read the WHOLE table and
# only sampled afterwards, which is backwards: sampling cannot save memory that
# has already been spent. 7,265,849 x 60 is 1.6 GB as float32, and the four
# string columns in y plus three in meta roughly double that once pandas'
# object arrays are counted.
#
# So: read y and meta with the columns actually needed, decide the split and
# the sample from those small tables, then stream X one row group at a time
# keeping ONLY the sampled rows. Peak becomes the sample, not the catalogue.
t0 = time.time()

_y_cols = set(pq.ParquetFile(DATASET_DIR / "y.parquet").schema_arrow.names)
_meta_cols = set(
    pq.ParquetFile(DATASET_DIR / "meta.parquet").schema_arrow.names
)

y = pd.read_parquet(
    DATASET_DIR / "y.parquet",
    columns=[c for c in ("label_weak", "label", "sample_weight")
             if c in _y_cols],
)
meta = pd.read_parquet(
    DATASET_DIR / "meta.parquet",
    columns=[c for c in ("event_id", "event_centroid_lon", "event_centroid_lat",
                         "start_time", "start_year")
             if c in _meta_cols],
)
print(f"loaded y {y.shape}  meta {meta.shape}  ({time.time() - t0:.1f}s)")
print("  y columns kept    :", list(y.columns))
print("  meta columns kept :", list(meta.columns))

# Hard gate on the feature NAMES, straight from the parquet footer, so it runs
# before a single row is read.
_X_COLS = list(pq.ParquetFile(DATASET_DIR / "X.parquet").schema_arrow.names)
db.assert_no_location_features(_X_COLS)
print("location check: PASS")
print(f"  X has {len(_X_COLS)} columns (from the footer, not a read)")

# Which label column to train on.
#
#   "weak"    the industrial-context hierarchy in `weak_labels`. It labels a
#             fifth of events instead of none, but it is MANUFACTURED from
#             features the model also sees, so every metric on it is optimistic
#             by construction. Model B excludes every rule input.
#   "strict"  the evidence-gated engine. Almost entirely UNKNOWN, so it cannot
#             train yet -- it needs the land-cover and weather groups.
LABEL_COLUMN = "weak" if "label_weak" in y.columns else "strict"

labels = (y["label_weak"] if LABEL_COLUMN == "weak" else y["label"]).to_numpy()
weights = (y["sample_weight"].to_numpy() if "sample_weight" in y.columns
           else np.ones(len(labels), dtype="float64"))

# `X` does not exist yet. It is filled AFTER the sample is chosen, straight
# into preallocated float32 arrays, one row group at a time. NaN is preserved,
# which is essential: NaN is how "could not measure" is encoded.

# Which class to fit. INDUSTRIAL_FLARE has only 3,571 events in the whole
# catalogue -- 894 in the training sample and 33 in the test set -- so its
# metrics are dominated by sampling noise and both models early-stopped at
# iteration 3. INDUSTRIAL_FIRE has 1,119,254 and is the first target that can
# actually support a model. Change this to compare.
TARGET = globals().get("TARGET", "INDUSTRIAL_FLARE")
print()
print(f"label column : {LABEL_COLUMN}")
if LABEL_COLUMN == "weak":
    print("  WEAK LABELS. These are manufactured from features the model sees,")
    print("  so PR-AUC here is an upper bound, NOT a generalisation estimate.")
    print("  Only an independent hand-labelled sample can give the latter.")
else:
    print("  strict labels -- nearly all UNKNOWN, so training cannot proceed.")
print()
print(f"target class : {TARGET}")
print("class counts (full catalogue):")
for cls in weak.WEAK_CLASSES:
    n = int((labels == cls).sum())
    print(f"  {cls:20s} {n:>10,}  {100.0 * n / max(len(labels), 1):>6.2f}%")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
split_kind = td.SPLIT_COMBINED
parts = td.make_split(meta, kind=split_kind, seed=0)
print(f"  split kind: {split_kind}")
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
print("  spatial leakage check: PASS (no shared block)")
if split_kind == td.SPLIT_OUT_OF_TIME:
    td.assert_no_time_leakage(meta[is_train], meta[is_test])
    print("  temporal leakage check: PASS (no test year in train)")
else:
    print("  temporal check: N/A -- the combined split trains on late events")
    print("    from non-held-out blocks by design; its guarantee is spatial.")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 2b. BOUND THE TRAINING SET
#
# LightGBM copies its input into a binary matrix, so a 7.27M x 60 float64 frame
# is roughly twice what the runtime has spare. Sampling is stratified on the
# TARGET so the minority class is not lost, and the sample size is printed so a
# metric is never mistaken for a full-catalogue result.
# ---------------------------------------------------------------------------

def stratified_sample(mask, cap, seed=0):
    """Indices from `mask`, at most `cap`, keeping the TARGET's prevalence."""
    idx = np.flatnonzero(mask)
    if len(idx) <= cap:
        return idx
    pos = idx[y_bin[idx] == 1]
    neg = idx[y_bin[idx] == 0]
    share = (len(pos) / len(idx)) if len(idx) else 0.0
    n_pos = min(len(pos), max(1, int(round(cap * share))))
    n_neg = min(len(neg), max(0, cap - n_pos))
    rng = np.random.default_rng(seed)
    take = np.concatenate([
        rng.choice(pos, n_pos, replace=False) if n_pos else np.empty(0, int),
        rng.choice(neg, n_neg, replace=False) if n_neg else np.empty(0, int),
    ])
    return np.sort(take)


train_idx = stratified_sample(is_train, MAX_TRAIN_ROWS, seed=0)
test_idx = stratified_sample(is_test, MAX_TEST_ROWS, seed=1)

print()
print("SAMPLE")
total_rows = pq.ParquetFile(DATASET_DIR / "X.parquet").metadata.num_rows
print(f"  train {len(train_idx):>9,} of {int(is_train.sum()):>10,}  "
      f"({100.0 * len(train_idx) / max(int(is_train.sum()), 1):.1f}%)")
print(f"  test  {len(test_idx):>9,} of {int(is_test.sum()):>10,}  "
      f"({100.0 * len(test_idx) / max(int(is_test.sum()), 1):.1f}%)")
print(f"  train positives {int(y_bin[train_idx].sum()):,}")
print(f"  test positives  {int(y_bin[test_idx].sum()):,}")
_full_gb = len(_X_COLS) * 4 * total_rows / 1024 ** 3
print(f"  a full float32 frame would be {_full_gb:.2f} GB; streaming avoids it")



# --- the partition must be the CURRENT one -----------------------------------
# A stale training_design.py computed model B as NON_LEAK - industrial, which
# leaves six weak-rule inputs (FRP_PERSISTENCE, DETECTION_COUNT_LOG,
# TIME_TO_PEAK, AREA_GROWTH_RATE, SPREAD_SPEED_MEDIAN, DIRECTION_CONSISTENCY)
# in the "scientific" model -- and those are the INDUSTRIAL_FIRE rule's own
# shape branch. That run reported 42 features and PR-AUC 0.527, both of which
# were artefacts. Fail loudly instead.
_model_a_cols, _model_b_cols, _leak = td.split_feature_sets(_X_COLS)
_expected_b = len(_model_b_cols)
_still_leaking = sorted(set(_model_b_cols) & set(weak.LABEL_INPUT_FEATURES))
if _still_leaking:
    raise SystemExit(
        "STALE training_design.py or weak_labels.py: these weak-rule inputs "
        f"are in model B: {_still_leaking}.\n"
        "Re-upload training_design.py and weak_labels.py, then re-run."
    )
print(f"  partition check: model A {len(_model_a_cols)}  model B "
      f"{_expected_b}  (rule inputs excluded: {len(_leak['excluded_from_b'])})")

non_leak = list(_model_b_cols)
label_derived = [c for c in _X_COLS if c in td.LABEL_DERIVED]
# ---------------------------------------------------------------------------
# 3. STREAM X -- keep only the sampled rows
#
# The catalogue is walked one row group at a time and only the sampled
# positions are retained, so peak memory is the SAMPLE rather than 7.27M rows.
# A full 60-column float32 frame is 1.6 GB; the sample is about 0.4 GB, and
# LightGBM's copy of it is smaller again.
# ---------------------------------------------------------------------------

_all = np.concatenate([train_idx, test_idx])
_all_sorted = np.sort(_all)
_keep_rows = _all_sorted

print()
print("streaming X, keeping only the sampled rows ...")
X_sampled = np.empty((len(_keep_rows), len(_X_COLS)), dtype="float32")
_filled = 0
_t0 = time.time()
_pf = pq.ParquetFile(DATASET_DIR / "X.parquet")
pf_rows = _pf.metadata.num_rows
_offset = 0
for _batch in _pf.iter_batches(batch_size=250_000, columns=_X_COLS):
    _n = _batch.num_rows
    _start = _offset
    _end = _offset + _n
    _lo = np.searchsorted(_keep_rows, _start)
    _hi = np.searchsorted(_keep_rows, _end)
    if _hi > _lo:
        local = _keep_rows[_lo:_hi] - _start
        block = _batch.to_pandas().to_numpy(dtype="float32", copy=True)
        X_sampled[_lo:_hi] = block[local]
        _filled += _hi - _lo
        del block
    _offset = _end
    if _offset % 1_000_000 < _n:
        print(f"    {_offset:>10,} / {pf_rows:,} rows  "
              f"({_filled:,} kept, {time.time() - _t0:.0f}s)")

print(f"  filled {_filled:,} of {len(_keep_rows):,} requested")
if _filled != len(_keep_rows):
    raise SystemExit(
        f"streaming fill got {_filled:,} rows, expected {len(_keep_rows):,}. "
        "The sample indices and the parquet row order must agree."
    )

_train_local = np.searchsorted(_keep_rows, train_idx)
_test_local = np.searchsorted(_keep_rows, test_idx)
_b_cols = [_X_COLS.index(c) for c in _model_b_cols]
X_train_b = np.ascontiguousarray(X_sampled[_train_local][:, _b_cols])
X_test_b = np.ascontiguousarray(X_sampled[_test_local][:, _b_cols])
X_train_a = np.ascontiguousarray(X_sampled[_train_local])
X_test_a = np.ascontiguousarray(X_sampled[_test_local])
del X_sampled

_mb = X_train_b.nbytes / 1024 ** 3
_ma = X_train_a.nbytes / 1024 ** 3
print(f"  model A train matrix {_ma:.2f} GB   model B train matrix {_mb:.2f} GB")
print("  (the 7.27M-row catalogue was never resident)")


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X_train_b, y_bin[train_idx], weights[train_idx],
    X_test_b, y_bin[test_idx], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[test_idx], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {len(_model_a_cols)} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X_train_a, y_bin[train_idx], weights[train_idx],
    X_test_a, y_bin[test_idx], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[test_idx], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

# Distance is only needed for the baseline, and only for the sampled
# rows, so it is read from the matrices already in memory.
_dist_col = _X_COLS.index("DIST_NEAREST_FACILITY_LOG")
_dist = np.concatenate([X_train_a[:, _dist_col], X_test_a[:, _dist_col]])
dist_m = np.where(np.isfinite(_dist), np.power(10.0, _dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    _dist[:len(train_idx)], y_bin[train_idx]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
print(f"PER-CLASS METRICS -- binary {TARGET} vs NOT_{TARGET},")
print("at the 0.60 review threshold. This model does NOT predict the other")
print("four classes; the full 6-class problem needs a multilabel head or six")
print("separate models, which is a later step.")
print("=" * 74)
for key in ("B", "A"):
    # This is a BINARY model (TARGET vs everything else), so reporting it in
    # six classes claimed that every non-TARGET row was predicted UNKNOWN --
    # which is a category it never saw. Only the target row and the
    # "everything else" row are actually being decided.
    pred_cls = np.full(len(y_bin[test_idx]), "NOT_" + TARGET, dtype=object)
    confident = prob_b if key == "B" else prob_a
    pred_cls[confident >= db.REVIEW_CONFIDENCE] = TARGET
    report = per_class_report(
        labels[test_idx], pred_cls, [TARGET, "NOT_" + TARGET],
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "label_column": LABEL_COLUMN,
    "label_caveat": (
        "labels are weak: manufactured from features the model sees. "
        "PR-AUC here is an UPPER BOUND, not a generalisation estimate. "
        "An independent hand-labelled sample is required to evaluate."
    ) if LABEL_COLUMN == "weak" else "strict evidence-gated labels",
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(_model_a_cols), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
# ---------------------------------------------------------------------------
# Persist the models themselves
#
# Without this the fitted boosters die with the runtime and the run has to be
# repeated to reproduce them. Written with the feature list and the exact
# target, because a booster is meaningless without both: it maps column
# POSITIONS, so reloading it against a reordered feature frame would silently
# score garbage.
# ---------------------------------------------------------------------------
import pickle

MODEL_DIR = QC_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for key, booster, cols in (
    ("A", models["A"], list(_model_a_cols)),
    ("B", models["B"], list(_model_b_cols)),
):
    path = MODEL_DIR / f"model_{key}.pkl"
    with path.open("wb") as fh:
        pickle.dump(
            {
                "booster": booster,
                "features": cols,
                "target": TARGET,
                "label_column": LABEL_COLUMN,
                "best_iteration": booster.best_iteration,
                "split": split_kind,
                "review_confidence": db.REVIEW_CONFIDENCE,
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
            },
            fh,
        )
    print(f"Wrote {path}  ({path.stat().st_size / 1024:.0f} KB, "
          f"{booster.best_iteration} trees)")

print()
print("Reloading requires the stored feature list, not the current X:")
print("  payload = pickle.load(open(...));")
print("  booster = payload['booster']; cols = payload['features']")
print("  prob = booster.predict(X[cols])")

ledger 1.3.0
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------
dataset: /content/drive/MyDrive/Firm_Clusters/full_2012_2026/features/dataset
loaded y (7265849, 3)  meta (7265849, 5)  (11.8s)
  y columns kept   

In [5]:
# ============================================================================
# P7 -- COLAB CELL: train and evaluate
#
# Consumes X / y / meta from P6 and reports what the brief asks for:
# precision, recall and F1 PER CLASS, on regions the model has never seen.
#
# Two models are always reported, because reporting only one of them is
# misleading in opposite directions:
#
#   model A  all available features, including the 18 label-derived industrial
#            ones. This is the operational model and it leans on facility
#            priors, so it is labelled as such.
#   model B  the non-leak partition only. This is the scientific claim.
#
# The distance-threshold baseline sits beside both. If neither model beats it,
# that is the finding.
# ============================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import colab_utils as cu
import dataset_builder as db
import feature_ledger as fl
import fire_labels as lab
import training_design as td
import weak_labels as weak

try:
    import lightgbm as lgb
except ImportError:
    print("=" * 74)
    print("LightGBM is not installed in this runtime.")
    print("Run:  !pip install -q lightgbm")
    print("Nothing below can proceed without it.")
    raise SystemExit(1)

DATASET_DIR = None   # filled in below from the resolved project root

print("=" * 74)
print(f"ledger {fl.LEDGER_VERSION}")
print("=" * 74)
print(cu.drive_report())

PROJECT_ROOT = cu.resolve_project_dir()
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"no project root under {cu.MYDRIVE}")
DATASET_DIR = PROJECT_ROOT / "full_2012_2026" / "features" / "dataset"
QC_DIR = PROJECT_ROOT / "qc"

for name in ("X.parquet", "y.parquet", "meta.parquet"):
    if not cu.path_ok(DATASET_DIR / name):
        raise FileNotFoundError(
            f"{DATASET_DIR / name} is missing. Run colab_p6_dataset.py first."
        )

print(f"dataset: {DATASET_DIR}")

# ---------------------------------------------------------------------------
# 1. LOAD
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 1. LOAD -- sampled, float32
#
# The first attempt loaded all 7,265,849 rows as float64 and then handed the
# frame to LightGBM, which copies it into its own binary matrix: roughly 3.5 GB
# for the frame plus another 3.5 GB inside LightGBM. That exhausted the runtime.
#
# Three fixes:
#   * read only the columns actually used
#   * cast to float32 -- half the bytes, and LightGBM wants float32 anyway
#   * train on a stratified sample, reported explicitly rather than silently
# ---------------------------------------------------------------------------

MAX_TRAIN_ROWS = int(globals().get("MAX_TRAIN_ROWS", 1_500_000))
MAX_TEST_ROWS = int(globals().get("MAX_TEST_ROWS", 600_000))

t0 = time.time()

# --- read ONLY what is needed, and stream X --------------------------------
# Two earlier versions exhausted the runtime. Both read the WHOLE table and
# only sampled afterwards, which is backwards: sampling cannot save memory that
# has already been spent. 7,265,849 x 60 is 1.6 GB as float32, and the four
# string columns in y plus three in meta roughly double that once pandas'
# object arrays are counted.
#
# So: read y and meta with the columns actually needed, decide the split and
# the sample from those small tables, then stream X one row group at a time
# keeping ONLY the sampled rows. Peak becomes the sample, not the catalogue.
t0 = time.time()

_y_cols = set(pq.ParquetFile(DATASET_DIR / "y.parquet").schema_arrow.names)
_meta_cols = set(
    pq.ParquetFile(DATASET_DIR / "meta.parquet").schema_arrow.names
)

y = pd.read_parquet(
    DATASET_DIR / "y.parquet",
    columns=[c for c in ("label_weak", "label", "sample_weight")
             if c in _y_cols],
)
meta = pd.read_parquet(
    DATASET_DIR / "meta.parquet",
    columns=[c for c in ("event_id", "event_centroid_lon", "event_centroid_lat",
                         "start_time", "start_year")
             if c in _meta_cols],
)
print(f"loaded y {y.shape}  meta {meta.shape}  ({time.time() - t0:.1f}s)")
print("  y columns kept    :", list(y.columns))
print("  meta columns kept :", list(meta.columns))

# Hard gate on the feature NAMES, straight from the parquet footer, so it runs
# before a single row is read.
_X_COLS = list(pq.ParquetFile(DATASET_DIR / "X.parquet").schema_arrow.names)
db.assert_no_location_features(_X_COLS)
print("location check: PASS")
print(f"  X has {len(_X_COLS)} columns (from the footer, not a read)")

# Which label column to train on.
#
#   "weak"    the industrial-context hierarchy in `weak_labels`. It labels a
#             fifth of events instead of none, but it is MANUFACTURED from
#             features the model also sees, so every metric on it is optimistic
#             by construction. Model B excludes every rule input.
#   "strict"  the evidence-gated engine. Almost entirely UNKNOWN, so it cannot
#             train yet -- it needs the land-cover and weather groups.
LABEL_COLUMN = "weak" if "label_weak" in y.columns else "strict"

labels = (y["label_weak"] if LABEL_COLUMN == "weak" else y["label"]).to_numpy()
weights = (y["sample_weight"].to_numpy() if "sample_weight" in y.columns
           else np.ones(len(labels), dtype="float64"))

# `X` does not exist yet. It is filled AFTER the sample is chosen, straight
# into preallocated float32 arrays, one row group at a time. NaN is preserved,
# which is essential: NaN is how "could not measure" is encoded.

# Which class to fit. INDUSTRIAL_FLARE has only 3,571 events in the whole
# catalogue -- 894 in the training sample and 33 in the test set -- so its
# metrics are dominated by sampling noise and both models early-stopped at
# iteration 3. INDUSTRIAL_FIRE has 1,119,254 and is the first target that can
# actually support a model. Change this to compare.
TARGET = globals().get("TARGET", "INDUSTRIAL_FLARE")
print()
print(f"label column : {LABEL_COLUMN}")
if LABEL_COLUMN == "weak":
    print("  WEAK LABELS. These are manufactured from features the model sees,")
    print("  so PR-AUC here is an upper bound, NOT a generalisation estimate.")
    print("  Only an independent hand-labelled sample can give the latter.")
else:
    print("  strict labels -- nearly all UNKNOWN, so training cannot proceed.")
print()
print(f"target class : {TARGET}")
print("class counts (full catalogue):")
for cls in weak.WEAK_CLASSES:
    n = int((labels == cls).sum())
    print(f"  {cls:20s} {n:>10,}  {100.0 * n / max(len(labels), 1):>6.2f}%")

# ---------------------------------------------------------------------------
# 2. SPLIT
# ---------------------------------------------------------------------------

meta = td.add_split_keys(meta)
split_kind = td.SPLIT_COMBINED
parts = td.make_split(meta, kind=split_kind, seed=0)
print(f"  split kind: {split_kind}")
train_ids = set(parts["train"]["event_id"])
test_ids = set(parts["test"]["event_id"])

is_train = meta["event_id"].isin(train_ids).to_numpy()
is_test = meta["event_id"].isin(test_ids).to_numpy()
dropped = ~(is_train | is_test)

print()
print("=" * 74)
print("SPLIT -- out-of-time AND out-of-space")
print("=" * 74)
print(f"  train : {is_train.sum():>10,}")
print(f"  test  : {is_test.sum():>10,}")
print(f"  dropped (early events inside held-out blocks): {dropped.sum():>10,}")
td.assert_no_spatial_leakage(meta[is_train], meta[is_test])
print("  spatial leakage check: PASS (no shared block)")
if split_kind == td.SPLIT_OUT_OF_TIME:
    td.assert_no_time_leakage(meta[is_train], meta[is_test])
    print("  temporal leakage check: PASS (no test year in train)")
else:
    print("  temporal check: N/A -- the combined split trains on late events")
    print("    from non-held-out blocks by design; its guarantee is spatial.")

y_bin = (labels == TARGET).astype(int)
print(f"  positives in train: {int(y_bin[is_train].sum()):,}")
print(f"  positives in test : {int(y_bin[is_test].sum()):,}")
if y_bin[is_train].sum() == 0 or y_bin[is_test].sum() == 0:
    raise SystemExit(
        "One side of the split has no positives. Widen the seed or lower the "
        "test_start_year before interpreting any metric."
    )


# ---------------------------------------------------------------------------
# 2b. BOUND THE TRAINING SET
#
# LightGBM copies its input into a binary matrix, so a 7.27M x 60 float64 frame
# is roughly twice what the runtime has spare. Sampling is stratified on the
# TARGET so the minority class is not lost, and the sample size is printed so a
# metric is never mistaken for a full-catalogue result.
# ---------------------------------------------------------------------------

def stratified_sample(mask, cap, seed=0):
    """Indices from `mask`, at most `cap`, keeping the TARGET's prevalence."""
    idx = np.flatnonzero(mask)
    if len(idx) <= cap:
        return idx
    pos = idx[y_bin[idx] == 1]
    neg = idx[y_bin[idx] == 0]
    share = (len(pos) / len(idx)) if len(idx) else 0.0
    n_pos = min(len(pos), max(1, int(round(cap * share))))
    n_neg = min(len(neg), max(0, cap - n_pos))
    rng = np.random.default_rng(seed)
    take = np.concatenate([
        rng.choice(pos, n_pos, replace=False) if n_pos else np.empty(0, int),
        rng.choice(neg, n_neg, replace=False) if n_neg else np.empty(0, int),
    ])
    return np.sort(take)


train_idx = stratified_sample(is_train, MAX_TRAIN_ROWS, seed=0)
test_idx = stratified_sample(is_test, MAX_TEST_ROWS, seed=1)

print()
print("SAMPLE")
total_rows = pq.ParquetFile(DATASET_DIR / "X.parquet").metadata.num_rows
print(f"  train {len(train_idx):>9,} of {int(is_train.sum()):>10,}  "
      f"({100.0 * len(train_idx) / max(int(is_train.sum()), 1):.1f}%)")
print(f"  test  {len(test_idx):>9,} of {int(is_test.sum()):>10,}  "
      f"({100.0 * len(test_idx) / max(int(is_test.sum()), 1):.1f}%)")
print(f"  train positives {int(y_bin[train_idx].sum()):,}")
print(f"  test positives  {int(y_bin[test_idx].sum()):,}")
_full_gb = len(_X_COLS) * 4 * total_rows / 1024 ** 3
print(f"  a full float32 frame would be {_full_gb:.2f} GB; streaming avoids it")



# --- the partition must be the CURRENT one -----------------------------------
# A stale training_design.py computed model B as NON_LEAK - industrial, which
# leaves six weak-rule inputs (FRP_PERSISTENCE, DETECTION_COUNT_LOG,
# TIME_TO_PEAK, AREA_GROWTH_RATE, SPREAD_SPEED_MEDIAN, DIRECTION_CONSISTENCY)
# in the "scientific" model -- and those are the INDUSTRIAL_FIRE rule's own
# shape branch. That run reported 42 features and PR-AUC 0.527, both of which
# were artefacts. Fail loudly instead.
_model_a_cols, _model_b_cols, _leak = td.split_feature_sets(_X_COLS)
_expected_b = len(_model_b_cols)
_still_leaking = sorted(set(_model_b_cols) & set(weak.LABEL_INPUT_FEATURES))
if _still_leaking:
    raise SystemExit(
        "STALE training_design.py or weak_labels.py: these weak-rule inputs "
        f"are in model B: {_still_leaking}.\n"
        "Re-upload training_design.py and weak_labels.py, then re-run."
    )
print(f"  partition check: model A {len(_model_a_cols)}  model B "
      f"{_expected_b}  (rule inputs excluded: {len(_leak['excluded_from_b'])})")

non_leak = list(_model_b_cols)
label_derived = [c for c in _X_COLS if c in td.LABEL_DERIVED]
# ---------------------------------------------------------------------------
# 3. STREAM X -- keep only the sampled rows
#
# The catalogue is walked one row group at a time and only the sampled
# positions are retained, so peak memory is the SAMPLE rather than 7.27M rows.
# A full 60-column float32 frame is 1.6 GB; the sample is about 0.4 GB, and
# LightGBM's copy of it is smaller again.
# ---------------------------------------------------------------------------

_all = np.concatenate([train_idx, test_idx])
_all_sorted = np.sort(_all)
_keep_rows = _all_sorted

print()
print("streaming X, keeping only the sampled rows ...")
X_sampled = np.empty((len(_keep_rows), len(_X_COLS)), dtype="float32")
_filled = 0
_t0 = time.time()
_pf = pq.ParquetFile(DATASET_DIR / "X.parquet")
pf_rows = _pf.metadata.num_rows
_offset = 0
for _batch in _pf.iter_batches(batch_size=250_000, columns=_X_COLS):
    _n = _batch.num_rows
    _start = _offset
    _end = _offset + _n
    _lo = np.searchsorted(_keep_rows, _start)
    _hi = np.searchsorted(_keep_rows, _end)
    if _hi > _lo:
        local = _keep_rows[_lo:_hi] - _start
        block = _batch.to_pandas().to_numpy(dtype="float32", copy=True)
        X_sampled[_lo:_hi] = block[local]
        _filled += _hi - _lo
        del block
    _offset = _end
    if _offset % 1_000_000 < _n:
        print(f"    {_offset:>10,} / {pf_rows:,} rows  "
              f"({_filled:,} kept, {time.time() - _t0:.0f}s)")

print(f"  filled {_filled:,} of {len(_keep_rows):,} requested")
if _filled != len(_keep_rows):
    raise SystemExit(
        f"streaming fill got {_filled:,} rows, expected {len(_keep_rows):,}. "
        "The sample indices and the parquet row order must agree."
    )

_train_local = np.searchsorted(_keep_rows, train_idx)
_test_local = np.searchsorted(_keep_rows, test_idx)
_b_cols = [_X_COLS.index(c) for c in _model_b_cols]
X_train_b = np.ascontiguousarray(X_sampled[_train_local][:, _b_cols])
X_test_b = np.ascontiguousarray(X_sampled[_test_local][:, _b_cols])
X_train_a = np.ascontiguousarray(X_sampled[_train_local])
X_test_a = np.ascontiguousarray(X_sampled[_test_local])
del X_sampled

_mb = X_train_b.nbytes / 1024 ** 3
_ma = X_train_a.nbytes / 1024 ** 3
print(f"  model A train matrix {_ma:.2f} GB   model B train matrix {_mb:.2f} GB")
print("  (the 7.27M-row catalogue was never resident)")


# ---------------------------------------------------------------------------
# 3. METRIC HELPERS -- per class, per the brief
# ---------------------------------------------------------------------------


def per_class_report(y_true, y_pred, classes):
    """Precision / recall / F1 per class. The brief asks for these explicitly."""
    rows = []
    for cls in classes:
        t = y_true == cls
        p = y_pred == cls
        tp = int(np.sum(t & p))
        fp = int(np.sum(~t & p))
        fn = int(np.sum(t & ~p))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) else 0.0)
        rows.append(
            {"class": cls, "support": int(t.sum()), "tp": tp, "fp": fp,
             "fn": fn, "precision": precision, "recall": recall, "f1": f1}
        )
    return pd.DataFrame(rows)


def fit_and_score(X_train, y_train, w_train, X_test, y_test, name):
    """Train one LightGBM model and score it. Returns (model, y_prob)."""
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 50,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "verbose": -1,
        "num_threads": -1,
        # NaN is meaningful here: 96.7% of events are single-cluster and 19 of
        # the 52 features are undefined for them. LightGBM routes NaN down a
        # default direction, which is exactly what "could not measure" should
        # do -- and is why no zero-filling is applied anywhere upstream.
        "zero_as_missing": False,
    }
    train_set = lgb.Dataset(
        X_train, label=y_train, weight=w_train, free_raw_data=False
    )
    valid_set = lgb.Dataset(
        X_test, label=y_test, reference=train_set, free_raw_data=False
    )
    model = lgb.train(
        params, train_set, num_boost_round=800,
        valid_sets=[valid_set],
        callbacks=[lgb.early_stopping(75, verbose=False),
                   lgb.log_evaluation(100)],
    )
    prob = model.predict(X_test, num_iteration=model.best_iteration)
    print(f"  {name}: best_iteration={model.best_iteration}")
    return model, prob


results = {}
models = {}

# ---------------------------------------------------------------------------
# 4. MODEL B -- the scientific claim (non-leak only)
# ---------------------------------------------------------------------------
print()
print("=" * 74)
print(f"model B (scientific): {len(non_leak)} non-leak features")
print(f"  excluded as label-derived: {label_derived}")
print("=" * 74)

model_b, prob_b = fit_and_score(
    X_train_b, y_bin[train_idx], weights[train_idx],
    X_test_b, y_bin[test_idx], "model B",
)
models["B"] = model_b
results["B"] = td.binary_metrics(y_bin[test_idx], (prob_b >= 0.5).astype(int))
results["B"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_b, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 5. MODEL A -- operational (all features, facility priors included)
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print(f"model A (operational): all {len(_model_a_cols)} features")
print(f"  includes the {len(label_derived)} label-derived industrial features")
print("=" * 74)

model_a, prob_a = fit_and_score(
    X_train_a, y_bin[train_idx], weights[train_idx],
    X_test_a, y_bin[test_idx], "model A",
)
models["A"] = model_a
results["A"] = td.binary_metrics(y_bin[test_idx], (prob_a >= 0.5).astype(int))
results["A"]["recall_at_target"] = td.precision_recall_at_precision(
    y_bin[test_idx], prob_a, db.REVIEW_CONFIDENCE
)

# ---------------------------------------------------------------------------
# 6. BASELINE -- the thing ML has to beat
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("BASELINE -- single threshold on DIST_NEAREST_FACILITY_LOG")
print("=" * 74)

# Distance is only needed for the baseline, and only for the sampled
# rows, so it is read from the matrices already in memory.
_dist_col = _X_COLS.index("DIST_NEAREST_FACILITY_LOG")
_dist = np.concatenate([X_train_a[:, _dist_col], X_test_a[:, _dist_col]])
dist_m = np.where(np.isfinite(_dist), np.power(10.0, _dist) - 1.0, np.nan)
# Tuned on TRAIN only. Selecting the threshold on the test fold and then
# reporting that fold's score is one of the quietest ways to overstate.
baseline = td.tune_distance_baseline(
    _dist[:len(train_idx)], y_bin[train_idx]
)
print(f"  threshold : {baseline['threshold_m']:,.0f} m  "
      f"(tuned on train)")
print(f"  precision : {baseline['precision']:.4f}")
print(f"  recall    : {baseline['recall']:.4f}")
print(f"  F1        : {baseline['f1']:.4f}")
print(f"  meets precision >= {td.TARGET_PRECISION}: "
      f"{baseline['meets_precision_target']}")

# ---------------------------------------------------------------------------
# 7. REPORT
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("RESULTS -- held-out regions, 2020+")
print("=" * 74)

for key in ("B", "A"):
    m = results[key]
    print()
    print(f"model {key}: PR-AUC {m['pr_auc']:.4f}   "
          f"F1 {m['f1']:.4f}   recall@prec>={db.REVIEW_CONFIDENCE} "
          f"{m['recall_at_target']['recall']:.4f}")
    verdict = td.compare_to_baseline(m, baseline)
    print(f"  vs baseline: {verdict['verdict']}")

print()
print("=" * 74)
# The negative side is the set of ACTUAL other classes, not a synthetic name.
# An earlier version reported a class literally called "NOT_INDUSTRIAL_FIRE",
# which never appears in the labels, so its support printed as 0 and the row
# looked like a class the model had never seen. Fold every non-target label
# into one real negative class so its support is the true count.
NEGATIVE = "OTHER_CLASS"
_test_labels = labels[test_idx]
_true_binary = np.where(_test_labels == TARGET, TARGET, NEGATIVE)

print(f"PER-CLASS METRICS -- binary {TARGET} vs {NEGATIVE},")
print(f"at the {db.REVIEW_CONFIDENCE} review threshold, on the held-out sample.")
print(f"{NEGATIVE} is the pooled remainder: "
      + ", ".join(
          f"{c} {int((_test_labels == c).sum()):,}"
          for c in weak.WEAK_CLASSES if c != TARGET and (_test_labels == c).sum()
      ))
print("This model does NOT separate the other classes from each other; the")
print("full 6-class problem needs a multilabel head or six models, later.")
print("=" * 74)
for key in ("B", "A"):
    confident = prob_b if key == "B" else prob_a
    pred_cls = np.where(
        confident >= db.REVIEW_CONFIDENCE, TARGET, NEGATIVE
    )
    report = per_class_report(
        _true_binary, pred_cls, [TARGET, NEGATIVE],
    )
    print()
    print(f"model {key}")
    print(report.to_string(index=False))
    # Confusion, so the table can be read without arithmetic.
    tp = int(((pred_cls == TARGET) & (_true_binary == TARGET)).sum())
    fp = int(((pred_cls == TARGET) & (_true_binary == NEGATIVE)).sum())
    fn = int(((pred_cls == NEGATIVE) & (_true_binary == TARGET)).sum())
    tn = int(((pred_cls == NEGATIVE) & (_true_binary == NEGATIVE)).sum())
    print(f"  confusion: tp={tp:,} fp={fp:,} fn={fn:,} tn={tn:,}")

# ---------------------------------------------------------------------------
# 8. REVIEW QUEUE -- the brief's 60% escalation
# ---------------------------------------------------------------------------

for key, prob in (("B", prob_b), ("A", prob_a)):
    queue = db.to_review_queue(prob, db.REVIEW_CONFIDENCE)
    n_review = int(queue["review"].sum())
    print()
    print(f"model {key}: {int(queue['accept'].sum()):,} auto-accept, "
          f"{n_review:,} to human review "
          f"({100.0 * n_review / len(prob):.1f}%)")

# ---------------------------------------------------------------------------
# 9. FEATURE IMPORTANCE -- does it look like behaviour or geography?
# ---------------------------------------------------------------------------

print()
print("=" * 74)
print("TOP FEATURES -- model B")
print("=" * 74)
importance = pd.DataFrame(
    {
        "feature": non_leak,
        "gain": models["B"].feature_importance("gain"),
        "splits": models["B"].feature_importance("split"),
    }
).sort_values("gain", ascending=False)
print(importance.head(20).to_string(index=False))

print()
print("=" * 74)
print("CHECK -- is the model leaning on location after all?")
print("=" * 74)
importance.to_csv(QC_DIR / "p7_feature_importance_B.csv", index=False)
leaked = [c for c in non_leak if db.is_location_column(c)]
print(f"  location columns in model B: {leaked or 'none'}")
top = importance.head(10)["feature"].tolist()
print(f"  top-10 features: {top}")
site_like = [c for c in top if c.startswith(("SITE_", "FRP_ANOMALY"))]
if site_like:
    print("  note: site-baseline features rank highly. That is permitted by the")
    print("  brief -- location enters through a site's history, not as a raw")
    print("  coordinate -- but report it rather than let it pass unremarked.")

# ---------------------------------------------------------------------------
# 10. PERSIST
# ---------------------------------------------------------------------------

QC_DIR.mkdir(parents=True, exist_ok=True)
payload = {
    "target": TARGET,
    "label_column": LABEL_COLUMN,
    "label_caveat": (
        "labels are weak: manufactured from features the model sees. "
        "PR-AUC here is an UPPER BOUND, not a generalisation estimate. "
        "An independent hand-labelled sample is required to evaluate."
    ) if LABEL_COLUMN == "weak" else "strict evidence-gated labels",
    "split": td.SPLIT_COMBINED,
    "n_train": int(is_train.sum()),
    "n_test": int(is_test.sum()),
    "model_b": {"features": non_leak, **results["B"]},
    "model_a": {"features": list(_model_a_cols), **results["A"]},
    "baseline": baseline,
    "review_confidence": db.REVIEW_CONFIDENCE,
}
(QC_DIR / "p7_results.json").write_text(json.dumps(payload, indent=2, default=str))
print()
print(f"Wrote {QC_DIR}/p7_results.json")
print(f"Wrote {QC_DIR}/p7_feature_importance_B.csv")
print()
# ---------------------------------------------------------------------------
# Persist the models themselves
#
# Without this the fitted boosters die with the runtime and the run has to be
# repeated to reproduce them. Written with the feature list and the exact
# target, because a booster is meaningless without both: it maps column
# POSITIONS, so reloading it against a reordered feature frame would silently
# score garbage.
# ---------------------------------------------------------------------------
import pickle

MODEL_DIR = QC_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for key, booster, cols in (
    ("A", models["A"], list(_model_a_cols)),
    ("B", models["B"], list(_model_b_cols)),
):
    path = MODEL_DIR / f"model_{key}.pkl"
    with path.open("wb") as fh:
        pickle.dump(
            {
                "booster": booster,
                "features": cols,
                "target": TARGET,
                "label_column": LABEL_COLUMN,
                "best_iteration": booster.best_iteration,
                "split": split_kind,
                "review_confidence": db.REVIEW_CONFIDENCE,
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
            },
            fh,
        )
    print(f"Wrote {path}  ({path.stat().st_size / 1024:.0f} KB, "
          f"{booster.best_iteration} trees)")

print()
print("Reloading requires the stored feature list, not the current X:")
print("  payload = pickle.load(open(...));")
print("  booster = payload['booster']; cols = payload['features']")
print("  prob = booster.predict(X[cols])")

ledger 1.3.0
DRIVE STATE
  /content/drive reachable : True
  MyDrive reachable       : True
  MyDrive entries         : ['Anamoly.ipynb', 'Colab Notebooks', 'Firm_Clusters', 'Project2', 'X_event_feature_inventory', 'feature_availability_analysis.md']
PROJECT ROOT CANDIDATES
--------------------------------------------------------------------------
  [no  ] /content/drive/MyDrive/Colab Notebooks  (0/2 required subpaths)
  [OK  ] /content/drive/MyDrive/Firm_Clusters  (2/2 required subpaths)
  [no  ] /content/drive/MyDrive/Project2  (0/2 required subpaths)
  [no  ] /content/drive/MyDrive/X_event_feature_inventory  (0/2 required subpaths)
--------------------------------------------------------------------------
resolved project root: /content/drive/MyDrive/Firm_Clusters
--------------------------------------------------------------------------
dataset: /content/drive/MyDrive/Firm_Clusters/full_2012_2026/features/dataset
loaded y (7265849, 3)  meta (7265849, 5)  (10.9s)
  y columns kept   